# Group 6 — LLM-primary fast pipeline: 9 target-specific calls

This version keeps the LLM at the centre of the pipeline and preserves the main mechanisms that improved the full alternative pipeline, while being designed to complete a full run in approximately **1–2 hours** under normal Nebius API performance.

## What is retained

- one separate LLM call for each of the 9 targets;
- target-specific guide;
- leakage-safe NMI + mRMR;
- CatBoost feature importance for feature selection;
- CatBoost probability prior in the prompt;
- 2 retrieved labelled respondents;
- structured probabilities;
- a small 12% probability blend;
- cache, checkpoints, and 100% coverage.

## What was removed for speed

- a separate latent-profile call;
- the standard reasoning second pass;
- 4 long retrieved examples;
- rerunning the legacy pipeline;
- ablations and multi-seed loops.

CatBoost is not the primary predictor. For valid API responses, the final probability is:

\[
P_{\mathrm{final}}=0.88P_{\mathrm{LLM}}+0.12P_{\mathrm{CatBoost}}.
\]

CatBoost is used as a fallback only when the LLM response remains technically invalid after one short repair call.

## Validation before the official test

The notebook loads the saved `pilot_ood_results_*.zip`, runs the new model on the same 75 OOD respondents, and compares it against:

1. the first Group 6 pipeline;
2. the full alternative pipeline.

Official inference is run only after confirming that the new version outperforms the first pipeline and retains a substantial share of the full alternative pipeline's improvement.

## Block 1. Install **libraries**

In [1]:
%pip install -q datasets openai numpy pandas scikit-learn catboost tqdm


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 10.0 MB/s eta 0:00:00


## Block 2. Imports

In [2]:
from __future__ import annotations

import copy
import hashlib
import json
import math
import os
import random
import re
import shutil
import threading
import time
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed
from dataclasses import asdict, dataclass, field
from getpass import getpass
from pathlib import Path
from typing import Any, Iterable, Optional

import numpy as np
from IPython.display import display
import pandas as pd
from datasets import load_dataset
try:
    from openai import OpenAI
except ImportError:
    OpenAI = None
from sklearn.metrics import accuracy_score, f1_score, normalized_mutual_info_score
from tqdm.auto import tqdm

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.max_rows", 200)

import zipfile
from datetime import datetime


## Block 3. Configuration

The main speed parameter is `max_workers=48`. If 429/rate-limit errors occur frequently, reduce it to 32. Increasing it above 48 is not recommended unless necessary.

In [3]:
@dataclass
class FastNineCallConfig:
    repo: str = "oxford-llms/ai-respondents-challenge"
    seed: int = 42
    strict_feature_only: bool = True

    # Exact split settings used in the saved OOD pilot
    in_domain_val_per_country: int = 30
    out_domain_n_countries: int = 15
    out_domain_val_per_country: int = 30

    # Leakage-safe selector
    min_availability: float = 0.25
    min_paired_rows: int = 100
    selection_pool_size: int = 40
    selected_features: int = 18
    redundancy_lambda: float = 0.20
    ood_region_penalty: float = 0.60
    mixed_region_penalty: float = 0.80

    # Hybrid feature-importance plan
    catboost_pool_features: int = 28
    protected_mrmr_features: int = 8
    catboost_importance_features: int = 4
    hybrid_selected_features: int = 16
    prompt_features: int = 12
    retrieval_features: int = 16
    selector_importance_weight: float = 0.70
    catboost_importance_weight: float = 0.20
    availability_weight: float = 0.10

    # Compact RAG
    use_retrieval: bool = True
    n_retrieval_examples: int = 2
    n_same_country_examples: int = 0
    max_examples_per_label: int = 1
    example_features: int = 3

    # CatBoost is an auxiliary expert, not the main method
    use_catboost: bool = True
    catboost_iterations: int = 180
    catboost_depth: int = 6
    catboost_learning_rate: float = 0.08
    catboost_in_prompt: bool = True
    catboost_blend_weight: float = 0.12

    # LLM
    model: str = "Qwen/Qwen3-32B"
    base_url: str = "https://api.studio.nebius.com/v1/"
    temperature: float = 0.0
    first_pass_max_tokens: int = 220
    repair_max_tokens: int = 220
    max_workers: int = 48
    retries: int = 2
    request_timeout_seconds: float = 120.0

    # Persistence
    cache_path: str = "llm_cache_fast9_strict_features_validation.jsonl"
    checkpoint_every: int = 100


CFG = FastNineCallConfig()
display(pd.DataFrame([asdict(CFG)]).T.rename(columns={0: "value"}))


,value
repo,oxford-llms/ai-respondents-challenge
seed,42
strict_feature_only,True
in_domain_val_per_country,30
out_domain_n_countries,15
out_domain_val_per_country,30
min_availability,0.25
min_paired_rows,100
selection_pool_size,40
selected_features,18


## Block 4. Load the official data

In [4]:
REPO = CFG.repo

train = load_dataset(REPO, "train", split="train").to_pandas()
test = load_dataset(REPO, "test", split="train").to_pandas()
targets = load_dataset(REPO, "targets", split="train").to_pandas()
features = load_dataset(REPO, "features", split="train").to_pandas()

print(f"Repository: {REPO}")
print(
    f"train {train.shape}, test {test.shape}, "
    f"{targets['question_id'].nunique()} targets, "
    f"{len(features)} allowed features"
)
print("train countries:", train["country"].nunique())
print("test countries:", test["country"].nunique())

required_target_columns = set(targets["question_id"].astype(str).unique())
missing_train_targets = required_target_columns - set(train.columns)
if missing_train_targets:
    raise ValueError(
        "Training data are missing target columns: "
        + ", ".join(sorted(missing_train_targets))
    )


README.md:   0%|          | 0.00/4.41k [00:00<?, ?B/s]

train.csv:   0%|          | 0.00/3.43M [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

test.csv:   0%|          | 0.00/628k [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

targets.csv:   0%|          | 0.00/5.02k [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

features.csv:   0%|          | 0.00/191k [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Repository: oxford-llms/ai-respondents-challenge
train (5000, 345), test (1050, 280), 9 targets, 278 allowed features
train countries: 50
test countries: 35


## Block 5. Connect to the Nebius API

In [5]:
if OpenAI is None:
    raise ImportError("Install the openai package first.")

nebius_key = os.getenv("NEBIUS_API_KEY", "").strip()
if not nebius_key:
    nebius_key = getpass("Nebius API key: ").strip()
if not nebius_key:
    raise ValueError("Nebius API key is empty.")

os.environ["NEBIUS_API_KEY"] = nebius_key

raw_nebius_client = OpenAI(
    base_url=CFG.base_url,
    api_key=nebius_key,
    timeout=CFG.request_timeout_seconds,
    max_retries=0,
)

print("Nebius client created.")


Nebius API key: ··········
Nebius client created.


## Block 6. Survey question and answer metadata

In [6]:
def canonical_code(value: Any) -> Optional[str]:
    if pd.isna(value):
        return None
    if isinstance(value, (np.integer, int)):
        return str(int(value))
    if isinstance(value, (np.floating, float)):
        number = float(value)
        if math.isfinite(number) and number.is_integer():
            return str(int(number))
        return format(number, "g")
    text = str(value).strip()
    try:
        number = float(text)
        if math.isfinite(number) and number.is_integer():
            return str(int(number))
        return format(number, "g")
    except (TypeError, ValueError):
        return text


def numeric_code(code: str):
    try:
        return float(code)
    except (TypeError, ValueError):
        return None


def sort_codes(codes: Iterable[str]):
    return sorted(
        codes,
        key=lambda value: (
            numeric_code(value) is None,
            numeric_code(value) if numeric_code(value) is not None else value,
        ),
    )


def parse_values_json(value: Any) -> dict[str, str]:
    if isinstance(value, dict):
        parsed = value
    elif pd.isna(value):
        parsed = {}
    else:
        try:
            parsed = json.loads(str(value))
        except json.JSONDecodeError:
            text = str(value)
            parsed = {}
            pair_pattern = re.compile(
                r'"((?:\\.|[^"\\])*)"\s*:\s*"((?:\\.|[^"\\])*)"'
            )
            for match in pair_pattern.finditer(text):
                try:
                    key = json.loads('"' + match.group(1) + '"')
                    label = json.loads('"' + match.group(2) + '"')
                    parsed[key] = label
                except Exception:
                    continue
    return {canonical_code(k): str(v) for k, v in parsed.items()}


qtext = dict(zip(features["variable"].astype(str), features["question"].astype(str)))
vmaps = {
    str(v): parse_values_json(m)
    for v, m in zip(features["variable"], features["values_json"])
}
question_for = (
    targets.drop_duplicates("question_id")
    .set_index("question_id")["question"]
    .astype(str)
    .to_dict()
)
labels_for = (
    targets.groupby("question_id", sort=False)["label"]
    .apply(lambda x: [str(v) for v in x])
    .to_dict()
)
target_ids = list(labels_for)
target_id_set = set(target_ids)

allowed_features = [
    str(v) for v in features["variable"]
    if str(v) in train.columns and str(v) in test.columns
]
candidate_features = [v for v in allowed_features if v not in target_id_set]

OFFICIAL_FEATURE_POOL = set(
    features["variable"].astype(str)
)

print("Targets:", target_ids)
print("Official feature variables:", len(OFFICIAL_FEATURE_POOL))
print("Allowed candidate features:", len(candidate_features))

assert len(OFFICIAL_FEATURE_POOL) == len(features), (
    "Duplicate variable codes found in the official features table."
)
assert set(candidate_features) <= OFFICIAL_FEATURE_POOL
assert set(candidate_features) <= set(test.columns.astype(str))
assert not (set(candidate_features) & target_id_set)
assert "country" not in candidate_features
assert "respondent_id" not in candidate_features

print(
    "Feature-pool gate passed: candidate_features are restricted "
    "to the official features dataset."
)


Targets: ['Q201', 'Q73', 'Q227', 'Q209', 'Q33', 'Q148', 'Q17', 'Q186', 'Q242']
Official feature variables: 278
Allowed candidate features: 278
Feature-pool gate passed: candidate_features are restricted to the official features dataset.


## Block 7. Convert target codes to official labels


In [7]:
def build_target_code_maps(train_df: pd.DataFrame, targets_df: pd.DataFrame):
    mappings: dict[str, dict[str, str]] = {}

    for target_id, group in targets_df.groupby("question_id", sort=False):
        target_id = str(target_id)
        ordered_labels = group["label"].astype(str).tolist()
        observed_codes = sort_codes(
            set(train_df[target_id].dropna().map(canonical_code).tolist())
        )

        if target_id == "Q186":
            expected = [
                "Never justifiable", "Rarely justifiable",
                "Sometimes justifiable", "Often justifiable",
                "Always justifiable",
            ]
            assert ordered_labels == expected
            mapping = {
                "1": expected[0], "2": expected[0],
                "3": expected[1], "4": expected[1],
                "5": expected[2], "6": expected[2],
                "7": expected[3], "8": expected[3],
                "9": expected[4], "10": expected[4],
            }
        elif target_id == "Q242":
            expected = [
                "It is against democracy", "Not essential",
                "Somewhat unimportant", "Moderately essential",
                "Essential", "Very essential",
            ]
            assert ordered_labels == expected
            mapping = {
                "0": expected[0],
                "1": expected[1], "2": expected[1],
                "3": expected[2], "4": expected[2],
                "5": expected[3], "6": expected[3],
                "7": expected[4], "8": expected[4],
                "9": expected[5], "10": expected[5],
            }
        else:
            mapping = {
                canonical_code(option): str(label)
                for option, label in zip(group["option"], group["label"])
            }

        missing = set(observed_codes) - set(mapping)
        if missing:
            raise ValueError(f"Unmapped target codes for {target_id}: {sort_codes(missing)}")
        mappings[target_id] = mapping

    return mappings


option_to_label = build_target_code_maps(train, targets)


def decode_target_value(target_id: str, value: Any) -> Optional[str]:
    code = canonical_code(value)
    if code is None:
        return None
    label = option_to_label[target_id].get(code)
    if label is None:
        raise ValueError(f"Unknown code {code!r} for {target_id}")
    return label


def decode_target_series(df: pd.DataFrame, target_id: str) -> pd.Series:
    return df[target_id].map(lambda v: decode_target_value(target_id, v))


for qid in target_ids:
    decoded = decode_target_series(train, qid)
    print(qid, decoded.value_counts(dropna=False).to_dict())


Q201 {'Never': 2007, 'Daily': 900, 'Weekly': 831, 'Less than monthly': 728, 'Monthly': 495, None: 39}
Q73 {'Not very much': 1733, 'None at all': 1527, 'Quite a lot': 1173, 'A great deal': 380, None: 187}
Q227 {'Fairly often': 1237, 'Not often': 1151, 'Very often': 1095, 'Not at all often': 992, None: 525}
Q209 {'Would never do': 2216, 'Might do': 1513, 'Have done': 1121, None: 150}
Q33 {'Disagree': 1566, 'Agree': 1098, 'Agree strongly': 866, 'Neither agree nor disagree': 797, 'Disagree strongly': 636, None: 37}
Q148 {'Very much': 1666, 'Not at all': 1003, 'A great deal': 1000, 'Not much': 951, None: 380}
Q17 {'Not so important': 3305, 'Important': 1616, None: 79}
Q186 {'Never justifiable': 1877, 'Always justifiable': 932, 'Sometimes justifiable': 883, 'Often justifiable': 496, 'Rarely justifiable': 476, None: 336}
Q242 {'Not essential': 1692, 'Moderately essential': 1097, 'Somewhat unimportant': 706, 'Very essential': 582, 'Essential': 569, None: 262, 'It is against democracy': 92}


## Блок 8. Leakage-safe splits

In [8]:
@dataclass
class ChallengeSplit:
    fit: pd.DataFrame
    valid: pd.DataFrame
    mode: str
    held_out_countries: list[str] = field(default_factory=list)
    seed: int = 42


def make_in_domain_split(
    df: pd.DataFrame,
    val_per_country: int = 30,
    seed: int = 42,
    min_fit_per_country: int = 20,
) -> ChallengeSplit:
    rng = np.random.default_rng(seed)
    valid_indices = []

    for country, group in df.groupby("country", sort=True):
        max_val = max(1, len(group) - min_fit_per_country)
        n_val = min(val_per_country, max_val)
        chosen = rng.choice(group.index.to_numpy(), size=n_val, replace=False)
        valid_indices.extend(chosen.tolist())

    valid_indices = sorted(valid_indices)
    valid = df.loc[valid_indices].copy().reset_index(drop=True)
    fit = df.drop(index=valid_indices).copy().reset_index(drop=True)

    assert set(valid["country"]).issubset(set(fit["country"]))
    assert set(valid["respondent_id"]).isdisjoint(set(fit["respondent_id"]))
    return ChallengeSplit(fit=fit, valid=valid, mode="in_domain", seed=seed)


def make_out_domain_split(
    df: pd.DataFrame,
    n_countries: int = 15,
    val_per_country: int = 30,
    seed: int = 42,
) -> ChallengeSplit:
    rng = np.random.default_rng(seed)
    countries = np.array(sorted(df["country"].dropna().unique()))
    if n_countries >= len(countries):
        raise ValueError("n_countries must be smaller than total train countries")

    held_out = sorted(rng.choice(countries, size=n_countries, replace=False).tolist())
    fit = df.loc[~df["country"].isin(held_out)].copy().reset_index(drop=True)

    valid_parts = []
    held_pool = df.loc[df["country"].isin(held_out)]
    for country, group in held_pool.groupby("country", sort=True):
        n = min(val_per_country, len(group))
        chosen = rng.choice(group.index.to_numpy(), size=n, replace=False)
        valid_parts.append(df.loc[chosen])

    valid = pd.concat(valid_parts, ignore_index=True)
    assert set(valid["country"]).isdisjoint(set(fit["country"]))
    assert set(valid["respondent_id"]).isdisjoint(set(fit["respondent_id"]))
    return ChallengeSplit(
        fit=fit,
        valid=valid,
        mode="out_of_domain",
        held_out_countries=held_out,
        seed=seed,
    )


def mask_target_columns(df: pd.DataFrame) -> pd.DataFrame:
    masked = df.copy()
    for target_id in target_ids:
        if target_id in masked.columns:
            masked[target_id] = np.nan
    return masked


in_split = make_in_domain_split(
    train,
    val_per_country=CFG.in_domain_val_per_country,
    seed=CFG.seed,
)
ood_split = make_out_domain_split(
    train,
    n_countries=CFG.out_domain_n_countries,
    val_per_country=CFG.out_domain_val_per_country,
    seed=CFG.seed,
)

print("IN-DOMAIN:", in_split.fit.shape, in_split.valid.shape)
print("OOD:", ood_split.fit.shape, ood_split.valid.shape)
print("OOD held-out countries:", ood_split.held_out_countries)


IN-DOMAIN: (3500, 345) (1500, 345)
OOD: (3500, 345) (450, 345)
OOD held-out countries: ['Bolivia', 'Brazil', 'Cyprus', 'Indonesia', 'Libya', 'Mongolia', 'Morocco', 'Peru', 'Romania', 'Serbia', 'Slovakia', 'Ukraine', 'United States', 'Uruguay', 'Vietnam']


## Block 9. NMI + mRMR selector


In [9]:
def safe_nmi(x: pd.Series, y: pd.Series) -> float:
    mask = x.notna() & y.notna()
    if mask.sum() < 2:
        return 0.0
    x2 = x.loc[mask].map(canonical_code).astype(str)
    y2 = y.loc[mask].astype(str)
    if x2.nunique() < 2 or y2.nunique() < 2:
        return 0.0
    return float(normalized_mutual_info_score(y2, x2))


class TargetFeatureSelector:
    def __init__(self, config: PipelineConfig):
        self.config = config
        self.rankings: dict[str, pd.DataFrame] = {}
        self.selected: dict[str, list[str]] = {}
        self.weights: dict[str, dict[str, float]] = {}
        self.fit_df: Optional[pd.DataFrame] = None
        self.mode: str = "mixed"
        self.feature_codes: dict[str, np.ndarray] = {}
        self.target_codes: dict[str, np.ndarray] = {}

    def _geo_penalty(self, variable: str) -> float:
        if variable != "N_REGION_ISO":
            return 1.0
        if self.mode == "out_of_domain":
            return self.config.ood_region_penalty
        if self.mode == "mixed":
            return self.config.mixed_region_penalty
        return 1.0

    @staticmethod
    def _factorize(series: pd.Series) -> np.ndarray:
        canonical = series.map(canonical_code)
        codes, _ = pd.factorize(canonical, sort=False)
        return codes.astype(np.int32, copy=False)  # missing values are -1

    @staticmethod
    def _nmi_from_codes(left: np.ndarray, right: np.ndarray, mask: np.ndarray) -> float:
        if int(mask.sum()) < 2:
            return 0.0
        x = left[mask]
        y = right[mask]
        if np.unique(x).size < 2 or np.unique(y).size < 2:
            return 0.0
        return float(normalized_mutual_info_score(y, x))

    def _prepare_codes(self):
        assert self.fit_df is not None
        self.feature_codes = {
            variable: self._factorize(self.fit_df[variable])
            for variable in tqdm(candidate_features, desc="Encoding feature columns")
        }
        self.target_codes = {}
        for target_id in target_ids:
            y = decode_target_series(self.fit_df, target_id)
            codes, _ = pd.factorize(y, sort=False)
            self.target_codes[target_id] = codes.astype(np.int32, copy=False)

    def _rank_one(self, target_id: str) -> pd.DataFrame:
        y_codes = self.target_codes[target_id]
        target_observed = y_codes >= 0
        target_rows = int(target_observed.sum())
        records = []

        for variable in candidate_features:
            x_codes = self.feature_codes[variable]
            paired = target_observed & (x_codes >= 0)
            n = int(paired.sum())
            if target_rows == 0:
                continue
            availability = n / target_rows
            if availability < self.config.min_availability or n < self.config.min_paired_rows:
                continue

            nmi = self._nmi_from_codes(x_codes, y_codes, paired)
            adjusted = nmi * math.sqrt(availability) * self._geo_penalty(variable)
            records.append({
                "feature": variable,
                "question": qtext.get(variable, variable),
                "nmi": nmi,
                "availability": availability,
                "paired_rows": n,
                "adjusted_score": adjusted,
            })

        if not records:
            raise ValueError(f"No eligible features for {target_id}")
        return (
            pd.DataFrame(records)
            .sort_values(["adjusted_score", "availability"], ascending=False)
            .reset_index(drop=True)
        )

    def _redundancy(self, left: str, right: str) -> float:
        x = self.feature_codes[left]
        y = self.feature_codes[right]
        mask = (x >= 0) & (y >= 0)
        if int(mask.sum()) < self.config.min_paired_rows:
            return 0.0
        return self._nmi_from_codes(x, y, mask)

    def _mrmr_select(self, ranking: pd.DataFrame) -> list[str]:
        pool = ranking.head(self.config.selection_pool_size).copy()
        relevance = dict(zip(pool["feature"], pool["adjusted_score"]))
        remaining = list(pool["feature"])
        selected: list[str] = []
        pair_cache: dict[tuple[str, str], float] = {}

        while remaining and len(selected) < self.config.selected_features:
            best_feature = None
            best_score = -np.inf

            for feature in remaining:
                if not selected:
                    redundancy = 0.0
                else:
                    vals = []
                    for chosen in selected:
                        key = tuple(sorted((feature, chosen)))
                        if key not in pair_cache:
                            pair_cache[key] = self._redundancy(feature, chosen)
                        vals.append(pair_cache[key])
                    redundancy = max(vals) if vals else 0.0

                score = relevance[feature] - self.config.redundancy_lambda * redundancy
                if score > best_score:
                    best_feature = feature
                    best_score = score

            selected.append(best_feature)
            remaining.remove(best_feature)

        return selected

    def fit(self, fit_df: pd.DataFrame, mode: str):
        self.fit_df = fit_df.reset_index(drop=True)
        self.mode = mode
        self.rankings = {}
        self.selected = {}
        self.weights = {}
        self._prepare_codes()

        for target_id in tqdm(target_ids, desc="Target feature selection"):
            ranking = self._rank_one(target_id)
            selected = self._mrmr_select(ranking)
            self.rankings[target_id] = ranking
            self.selected[target_id] = selected

            raw_weights = (
                ranking.set_index("feature")["adjusted_score"]
                .reindex(selected)
                .fillna(0.0)
            )
            scale = float(raw_weights.max()) or 1.0
            self.weights[target_id] = (raw_weights / scale).to_dict()
        return self

    def available_prompt_features(self, row: pd.Series, target_id: str) -> list[str]:
        ordered = self.rankings[target_id]["feature"].tolist()
        available = [v for v in ordered if not pd.isna(row.get(v))]
        return available[: self.config.prompt_features]


## Block 10. Fit-only priors and CatBoost expert


In [10]:
def compute_fit_priors(fit_df: pd.DataFrame):
    majority = {}
    priors = {}
    for target_id in target_ids:
        y = decode_target_series(fit_df, target_id).dropna()
        shares = y.value_counts(normalize=True).reindex(labels_for[target_id], fill_value=0.0)
        majority[target_id] = str(shares.idxmax())
        priors[target_id] = shares.to_dict()
    return majority, priors


MISSING_TOKEN = "__MISSING__"


def categorical_frame(df: pd.DataFrame, variables: list[str]) -> pd.DataFrame:
    out = pd.DataFrame(index=df.index)
    for variable in variables:
        out[variable] = df[variable].map(canonical_code).fillna(MISSING_TOKEN).astype(str)
    return out


class CatBoostExpert:
    def __init__(self, config: FastNineCallConfig):
        self.config = config
        self.models = {}
        self.features = {}
        self.constants = {}
        self.available = False

    def fit(self, fit_df: pd.DataFrame, selected_features: dict[str, list[str]]):
        if not self.config.use_catboost:
            return self
        try:
            from catboost import CatBoostClassifier
        except Exception as exc:
            print("CatBoost unavailable; continuing without it:", exc)
            return self

        for target_id in target_ids:
            y = decode_target_series(fit_df, target_id)
            mask = y.notna()
            y_fit = y.loc[mask].astype(str)
            vars_ = selected_features[target_id][: self.config.retrieval_features]
            self.features[target_id] = vars_

            if y_fit.nunique() < 2:
                self.constants[target_id] = str(y_fit.mode().iloc[0])
                continue

            X = categorical_frame(fit_df.loc[mask], vars_)
            loss = "Logloss" if y_fit.nunique() == 2 else "MultiClass"
            model = CatBoostClassifier(
                iterations=self.config.catboost_iterations,
                depth=self.config.catboost_depth,
                learning_rate=self.config.catboost_learning_rate,
                loss_function=loss,
                auto_class_weights="Balanced",
                random_seed=self.config.seed,
                verbose=False,
                allow_writing_files=False,
                thread_count=-1,
            )
            model.fit(X, y_fit, cat_features=list(range(X.shape[1])))
            self.models[target_id] = model

        self.available = bool(self.models or self.constants)
        return self

    def predict_proba(self, row: pd.Series, target_id: str) -> Optional[dict[str, float]]:
        labels = labels_for[target_id]
        if target_id in self.constants:
            result = {label: 0.0 for label in labels}
            result[self.constants[target_id]] = 1.0
            return result
        model = self.models.get(target_id)
        if model is None:
            return None

        vars_ = self.features[target_id]
        X = categorical_frame(pd.DataFrame([row]), vars_)
        probs = model.predict_proba(X)[0]
        result = {label: 0.0 for label in labels}
        for cls, prob in zip(model.classes_, probs):
            if str(cls) in result:
                result[str(cls)] = float(prob)
        total = sum(result.values())
        return {k: v / total for k, v in result.items()} if total > 0 else None


## Блок 11. Retrieval похожих train-респондентов

In [11]:
class SimilarRespondentRetriever:
    """
    Global retrieval using only official feature variables.

    `country` and `respondent_id` never enter the similarity calculation.
    respondent_id is retained only to identify the retrieved training row
    in diagnostics.
    """

    def __init__(
        self,
        fit_df: pd.DataFrame,
        target_id: str,
        variables: list[str],
        weights: dict[str, float],
        config: FastNineCallConfig,
    ):
        self.target_id = target_id
        self.variables = list(variables)
        self.config = config

        assert set(self.variables) <= OFFICIAL_FEATURE_POOL
        assert "country" not in self.variables
        assert "respondent_id" not in self.variables
        assert not (set(self.variables) & target_id_set)

        y = decode_target_series(
            fit_df,
            target_id,
        )
        mask = y.notna()

        self.df = (
            fit_df.loc[mask]
            .copy()
            .reset_index(drop=True)
        )
        self.labels = (
            y.loc[mask]
            .astype(str)
            .reset_index(drop=True)
        )
        self.respondent_ids = (
            self.df["respondent_id"]
            .astype(str)
            .to_numpy()
        )

        X = categorical_frame(
            self.df,
            self.variables,
        )
        self.matrix = X.to_numpy(
            dtype=object,
        )
        self.weights = np.array([
            max(
                float(weights.get(variable, 0.05)),
                0.02,
            )
            for variable in self.variables
        ])

    def _scores(
        self,
        row: pd.Series,
    ) -> np.ndarray:
        query = np.array([
            (
                canonical_code(row.get(variable))
                if not pd.isna(row.get(variable))
                else MISSING_TOKEN
            )
            for variable in self.variables
        ], dtype=object)

        query[query == None] = MISSING_TOKEN  # noqa: E711

        valid = (
            (self.matrix != MISSING_TOKEN)
            & (query[None, :] != MISSING_TOKEN)
        )
        denominator = (
            valid * self.weights[None, :]
        ).sum(axis=1)

        matches = (
            (self.matrix == query[None, :])
            & valid
        )
        numerator = (
            matches * self.weights[None, :]
        ).sum(axis=1)

        coverage = valid.mean(axis=1)

        score = np.divide(
            numerator,
            denominator,
            out=np.zeros_like(
                numerator,
                dtype=float,
            ),
            where=denominator > 0,
        )

        return score + 0.03 * coverage

    def _choose_diverse(
        self,
        indices: np.ndarray,
        scores: np.ndarray,
        n: int,
        banned: set[int],
    ) -> list[int]:
        chosen = []
        label_counts = Counter()

        ordered = indices[
            np.argsort(
                scores[indices]
            )[::-1]
        ]

        for index in ordered:
            index = int(index)

            if index in banned:
                continue

            label = self.labels.iloc[index]

            if (
                label_counts[label]
                >= self.config.max_examples_per_label
            ):
                continue

            chosen.append(index)
            label_counts[label] += 1

            if len(chosen) >= n:
                break

        return chosen

    def retrieve(
        self,
        row: pd.Series,
        row_mode: Optional[str] = None,
    ) -> list[dict[str, Any]]:
        """
        Retrieve globally from fit data.

        row_mode is accepted only for interface compatibility and is ignored.
        No country-specific filtering is performed.
        """
        if not self.config.use_retrieval:
            return []

        scores = self._scores(row)
        all_indices = np.arange(
            len(self.df)
        )

        chosen = self._choose_diverse(
            indices=all_indices,
            scores=scores,
            n=self.config.n_retrieval_examples,
            banned=set(),
        )

        examples = []

        for index in chosen:
            examples.append({
                "respondent": self.df.iloc[index],
                "target_label": self.labels.iloc[index],
                "similarity": float(
                    scores[index]
                ),
                "respondent_id": (
                    self.respondent_ids[index]
                ),
            })

        return examples


## Блок 12. Target-specific guides

In [12]:
TARGET_GUIDES = {
    "Q201": (
        "Infer habitual information-source use. Give most weight to the respondent's use of "
        "other news and communication channels, general media engagement, education and age. "
        "Distinguish daily, weekly, occasional and never use rather than merely online/offline preference."
    ),
    "Q73": (
        "Infer latent institutional trust. Give strongest weight to confidence in government, political parties, "
        "elections, courts, civil service and police. Use country only as weak contextual evidence."
    ),
    "Q227": (
        "Infer the respondent's perception of electoral integrity. Prioritise answers about vote buying, threats, "
        "fair counting, media fairness, corruption and political trust."
    ),
    "Q209": (
        "Infer willingness to participate in non-electoral politics. Prioritise past or potential participation in "
        "petitions, boycotts, demonstrations, online activism and civic organisations."
    ),
    "Q33": (
        "Infer gender-role traditionalism concerning employment. Prioritise views on women's leadership, education, "
        "family roles, motherhood and equality. Do not treat country as determinative."
    ),
    "Q148": (
        "Infer perceived personal or national security risk. Prioritise worries about war, terrorism, political conflict, "
        "economic insecurity and neighbourhood safety. Distinguish high concern from a generally negative worldview."
    ),
    "Q17": (
        "Infer preference for child obedience versus autonomy. Consider religion, authority orientation, family values, "
        "traditionalism and whether independence is valued for children."
    ),
    "Q186": (
        "Infer sexual-morality permissiveness. Prioritise views on divorce, homosexuality, abortion, prostitution, "
        "casual sex, religiosity and social conservatism. Map the inferred 1-10 position to the supplied bins."
    ),
    "Q242": (
        "Infer support for religious authority in democratic law. Consider religiosity, church-state preferences, "
        "democratic values, authoritarianism and confidence in religious institutions. Respect the special anti-democratic category."
    ),
}

LABEL_NOTES = {
    "Q186": (
        "Official bins: Never justifiable = raw 1-2; Rarely justifiable = 3-4; "
        "Sometimes justifiable = 5-6; Often justifiable = 7-8; Always justifiable = 9-10."
    ),
    "Q242": (
        "Official bins: It is against democracy = special raw response 0; Not essential = 1-2; "
        "Somewhat unimportant = 3-4; Moderately essential = 5-6; Essential = 7-8; Very essential = 9-10."
    ),
}

PROFILE_DIMENSIONS = [
    "institutional_trust",
    "political_participation",
    "electoral_integrity_concern",
    "gender_traditionalism",
    "security_anxiety",
    "religiosity",
    "authority_orientation",
    "social_conservatism",
    "media_engagement",
]


## Блок 13. Cache, parsing и probability helpers

In [13]:
class CachedNebiusClient:
    def __init__(
        self,
        config: PipelineConfig,
        api_key: Optional[str] = None,
        client: Optional[Any] = None,
    ):
        self.config = config
        if OpenAI is None:
            raise ImportError("Install the openai package first: pip install openai")

        if client is not None:
            self.client = client
        else:
            api_key = api_key or os.getenv("NEBIUS_API_KEY")
            if not api_key:
                api_key = getpass("Nebius API key: ").strip()
            if not api_key:
                raise ValueError("Nebius API key is empty")
            os.environ["NEBIUS_API_KEY"] = api_key
            self.client = OpenAI(base_url=config.base_url, api_key=api_key)
        self.cache_path = Path(config.cache_path)
        self.lock = threading.Lock()
        self.cache: dict[str, str] = {}
        if self.cache_path.exists():
            with self.cache_path.open("r", encoding="utf-8") as f:
                for line in f:
                    try:
                        record = json.loads(line)
                        self.cache[record["key"]] = record["response"]
                    except Exception:
                        pass
        print(f"Loaded {len(self.cache)} cached LLM responses")

    def _key(self, system: str, user: str, thinking: bool, max_tokens: int) -> str:
        payload = json.dumps({
            "model": self.config.model,
            "system": system,
            "user": user,
            "thinking": thinking,
            "max_tokens": max_tokens,
            "temperature": self.config.temperature,
        }, ensure_ascii=False, sort_keys=True)
        return hashlib.sha256(payload.encode("utf-8")).hexdigest()

    def complete(self, system: str, user: str, thinking: bool = False, max_tokens: int = 320) -> str:
        key = self._key(system, user, thinking, max_tokens)
        with self.lock:
            cached = self.cache.get(key)
        if cached is not None:
            return cached

        last_error = None
        for attempt in range(self.config.retries):
            try:
                response = self.client.chat.completions.create(
                    model=self.config.model,
                    messages=[
                        {"role": "system", "content": system},
                        {"role": "user", "content": user},
                    ],
                    temperature=self.config.temperature,
                    max_tokens=max_tokens,
                    extra_body={"chat_template_kwargs": {"enable_thinking": thinking}},
                )
                text = response.choices[0].message.content or ""
                with self.lock:
                    if key not in self.cache:
                        self.cache[key] = text
                        with self.cache_path.open("a", encoding="utf-8") as f:
                            f.write(json.dumps({"key": key, "response": text}, ensure_ascii=False) + "\n")
                return text
            except Exception as exc:
                last_error = exc
                if attempt < self.config.retries - 1:
                    time.sleep(min(2 ** attempt, 10) + random.random())
        raise last_error


def normalise_label(text: Any, labels: list[str]) -> Optional[str]:
    cleaned = re.sub(r"\s+", " ", str(text).strip().strip('"').strip("'"))
    for label in labels:
        if cleaned == label or cleaned.casefold() == label.casefold():
            return label
    matches = [label for label in labels if label.casefold() in cleaned.casefold()]
    if len(matches) == 1:
        return matches[0]
    if len(matches) > 1:
        matches = sorted(matches, key=len, reverse=True)
        if len(matches[0]) > len(matches[1]):
            return matches[0]
    return None


def extract_json_object(text: str) -> Optional[dict[str, Any]]:
    cleaned = re.sub(r"```(?:json)?", "", str(text), flags=re.I).replace("```", "").strip()
    try:
        obj = json.loads(cleaned)
        return obj if isinstance(obj, dict) else None
    except Exception:
        pass
    start, end = cleaned.find("{"), cleaned.rfind("}")
    if start >= 0 and end > start:
        try:
            obj = json.loads(cleaned[start:end + 1])
            return obj if isinstance(obj, dict) else None
        except Exception:
            return None
    return None


def parse_probability_response(text: str, labels: list[str]):
    obj = extract_json_object(text)
    prediction = None
    probs = {label: 0.0 for label in labels}

    if obj is not None:
        prediction = normalise_label(obj.get("prediction"), labels)
        raw_probs = obj.get("probabilities", {})
        if isinstance(raw_probs, dict):
            for raw_label, raw_prob in raw_probs.items():
                label = normalise_label(raw_label, labels)
                if label is None:
                    continue
                try:
                    probs[label] = max(float(raw_prob), 0.0)
                except Exception:
                    pass

    if prediction is None:
        prediction = normalise_label(text, labels)

    total = sum(probs.values())
    if total > 0:
        probs = {k: v / total for k, v in probs.items()}
    elif prediction is not None:
        probs[prediction] = 1.0
    else:
        return None

    if prediction is None:
        prediction = max(probs, key=probs.get)
    return {"prediction": prediction, "probabilities": probs}


def probability_margin(probs: dict[str, float]) -> float:
    values = sorted(probs.values(), reverse=True)
    return values[0] - values[1] if len(values) > 1 else 1.0


## Block 14. Evidence rendering

In [14]:
def answer_text(variable: str, value: Any) -> Optional[str]:
    code = canonical_code(value)
    if code is None:
        return None
    mapping = vmaps.get(variable, {})
    if variable == "N_REGION_ISO" and code not in mapping:
        return None
    return mapping.get(code, code)


def render_evidence(
    row: pd.Series,
    variables: list[str],
    max_items: Optional[int] = None,
) -> str:
    blocks = []
    for variable in variables:
        value = row.get(variable)
        if pd.isna(value):
            continue
        decoded = answer_text(variable, value)
        if decoded is None:
            continue
        blocks.append(
            f"- [{variable}] {qtext.get(variable, variable)}\n"
            f"  Answer: {decoded}"
        )
        if max_items is not None and len(blocks) >= max_items:
            break
    return "\n".join(blocks) if blocks else "No usable answers available."


## Block 15. Hybrid feature importance

For each target:

1. CatBoost is trained on an expanded top-28 NMI/mRMR pool.
2. The first 8 mRMR features are protected from being dropped.
3. Up to 4 of the most important CatBoost features are added.
4. The remaining slots up to 16 are filled using the hybrid score:

\[
0.70\,\text{selector score}+0.20\,\text{CatBoost importance}+0.10\,\text{availability}.
\]

The prompt receives 12 available features from this target-specific list.

In [15]:
def catboost_feature_importance(
    expert: CatBoostExpert,
    target_id: str,
) -> dict[str, float]:
    model = expert.models.get(target_id)
    variables = expert.features.get(target_id, [])

    if model is None or not variables:
        return {variable: 0.0 for variable in variables}

    raw = model.get_feature_importance()
    result = {
        variable: max(float(value), 0.0)
        for variable, value in zip(variables, raw)
    }
    scale = max(result.values(), default=0.0)

    if scale > 0:
        result = {
            variable: value / scale
            for variable, value in result.items()
        }
    return result


def build_hybrid_feature_plan(
    selector: TargetFeatureSelector,
    expert: CatBoostExpert,
    config: FastNineCallConfig,
):
    hybrid_rankings = {}
    hybrid_selected = {}
    hybrid_weights = {}
    importance_rows = []

    for target_id in target_ids:
        ranking = selector.rankings[target_id].copy()

        selector_max = float(
            ranking["adjusted_score"].max()
        ) or 1.0
        ranking["selector_norm"] = (
            ranking["adjusted_score"] / selector_max
        )

        cb_map = catboost_feature_importance(
            expert,
            target_id,
        )
        ranking["catboost_importance"] = (
            ranking["feature"]
            .map(cb_map)
            .fillna(0.0)
        )

        ranking["hybrid_score"] = (
            config.selector_importance_weight
            * ranking["selector_norm"]
            + config.catboost_importance_weight
            * ranking["catboost_importance"]
            + config.availability_weight
            * ranking["availability"].clip(0.0, 1.0)
        )

        ranking = ranking.sort_values(
            ["hybrid_score", "adjusted_score", "availability"],
            ascending=False,
        ).reset_index(drop=True)

        protected = list(
            selector.selected[target_id][
                : config.protected_mrmr_features
            ]
        )

        cb_top = [
            feature
            for feature, value in sorted(
                cb_map.items(),
                key=lambda pair: pair[1],
                reverse=True,
            )
            if value > 0
        ][: config.catboost_importance_features]

        selected = []
        for feature in protected + cb_top + ranking["feature"].tolist():
            if feature not in selected:
                selected.append(feature)
            if len(selected) >= config.hybrid_selected_features:
                break

        hybrid_rankings[target_id] = ranking
        hybrid_selected[target_id] = selected

        scores = (
            ranking.set_index("feature")["hybrid_score"]
            .reindex(selected)
            .fillna(0.0)
        )
        score_scale = float(scores.max()) or 1.0
        hybrid_weights[target_id] = (
            scores / score_scale
        ).to_dict()

        selected_set = set(selected)
        protected_set = set(protected)
        cb_top_set = set(cb_top)

        for _, item in ranking.iterrows():
            importance_rows.append({
                "question_id": target_id,
                "feature": item["feature"],
                "question": item["question"],
                "selected_for_prompt_or_retrieval": (
                    item["feature"] in selected_set
                ),
                "protected_mrmr": (
                    item["feature"] in protected_set
                ),
                "catboost_top_addition": (
                    item["feature"] in cb_top_set
                ),
                "nmi": float(item["nmi"]),
                "availability": float(item["availability"]),
                "adjusted_score": float(item["adjusted_score"]),
                "selector_norm": float(item["selector_norm"]),
                "catboost_importance": float(
                    item["catboost_importance"]
                ),
                "hybrid_score": float(item["hybrid_score"]),
            })

    return (
        hybrid_rankings,
        hybrid_selected,
        hybrid_weights,
        pd.DataFrame(importance_rows),
    )


## Block 16. LLM-primary fast pipeline

The pipeline still makes 9 separate target-specific calls for each respondent. A repair call is used only when the first response cannot be parsed.

In [16]:
PREDICTION_SYSTEM = """
You are a calibrated survey-response prediction model.
Predict how this specific real respondent answered the target survey question.
This is a classification task, not a request for your own opinion.

Use only:
- respondent answers drawn from the official allowed feature variables;
- retrieved fit respondents selected using those same allowed variables;
- the auxiliary CatBoost probability prior.

Do not infer the answer from country, respondent identity, or any variable
outside the official features dataset. Retrieved respondents are analogies,
not rules. The CatBoost prior is supporting evidence and must not replace
respondent-specific inference.

Return valid JSON only and assign probabilities to every allowed label.
""".strip()


REPAIR_SYSTEM = """
You repair structured classification output.
Return valid JSON only. Do not add explanation or markdown.
""".strip()


class FastNineCallLLMPipeline:
    def __init__(
        self,
        config: FastNineCallConfig,
        llm: CachedNebiusClient,
    ):
        self.config = config
        self.llm = llm
        self.fit_df = None
        self.mode = "mixed"
        self.selector = TargetFeatureSelector(config)
        self.majority = {}
        self.priors = {}
        self.expert = CatBoostExpert(config)

        self.hybrid_rankings = {}
        self.hybrid_selected = {}
        self.hybrid_weights = {}
        self.feature_importance_table = pd.DataFrame()

        self.retrievers = {}

    def fit(
        self,
        fit_df: pd.DataFrame,
        mode: str = "mixed",
    ):
        self.fit_df = fit_df.copy().reset_index(drop=True)
        self.mode = mode
        started = time.perf_counter()

        self.selector.fit(
            self.fit_df,
            mode=mode,
        )
        self.majority, self.priors = compute_fit_priors(
            self.fit_df
        )

        catboost_pool = {}
        for target_id in target_ids:
            pool = (
                self.selector.selected[target_id]
                + self.selector.rankings[target_id]
                    .head(self.config.catboost_pool_features)[
                        "feature"
                    ]
                    .tolist()
            )
            catboost_pool[target_id] = list(
                dict.fromkeys(pool)
            )[: self.config.catboost_pool_features]

        self.expert.fit(
            self.fit_df,
            catboost_pool,
        )

        (
            self.hybrid_rankings,
            self.hybrid_selected,
            self.hybrid_weights,
            self.feature_importance_table,
        ) = build_hybrid_feature_plan(
            self.selector,
            self.expert,
            self.config,
        )

        self.retrievers = {}
        for target_id in target_ids:
            variables = self.hybrid_selected[target_id][
                : self.config.retrieval_features
            ]
            self.retrievers[target_id] = (
                SimilarRespondentRetriever(
                    fit_df=self.fit_df,
                    target_id=target_id,
                    variables=variables,
                    weights=self.hybrid_weights[target_id],
                    config=self.config,
                )
            )

        elapsed = (time.perf_counter() - started) / 60
        print(
            f"Pipeline fit completed in {elapsed:.2f} minutes."
        )
        return self

    def row_mode(self, row: pd.Series) -> str:
        # Country never changes predictive behaviour.
        return "global"

    def available_prompt_features(
        self,
        row: pd.Series,
        target_id: str,
    ) -> list[str]:
        ranking_order = self.hybrid_rankings[
            target_id
        ]["feature"].tolist()

        ordered = list(
            dict.fromkeys(
                self.hybrid_selected[target_id]
                + ranking_order
            )
        )

        available = [
            variable
            for variable in ordered
            if not pd.isna(row.get(variable))
            and answer_text(variable, row.get(variable)) is not None
        ]

        return available[
            : self.config.prompt_features
        ]

    def _catboost_probs(
        self,
        row: pd.Series,
        target_id: str,
    ) -> dict[str, float]:
        probabilities = (
            self.expert.predict_proba(row, target_id)
            if self.expert.available
            else None
        )

        if probabilities is None:
            probabilities = dict(
                self.priors[target_id]
            )

        total = sum(probabilities.values())
        if total <= 0:
            probabilities = dict(
                self.priors[target_id]
            )
            total = sum(probabilities.values())

        return {
            label: float(
                probabilities.get(label, 0.0)
            ) / total
            for label in labels_for[target_id]
        }

    def _render_examples(
        self,
        examples: list[dict[str, Any]],
        target_id: str,
    ) -> str:
        if not examples:
            return "No retrieved examples used."

        variables = self.hybrid_selected[target_id]
        blocks = []

        for number, example in enumerate(
            examples,
            start=1,
        ):
            evidence = render_evidence(
                example["respondent"],
                variables,
                max_items=self.config.example_features,
            )
            blocks.append(
                f"Example {number} "
                f"(similarity={example['similarity']:.3f}):\n"
                f"{evidence}\n"
                f"Observed target answer: "
                f"{example['target_label']}"
            )

        return "\n\n".join(blocks)

    def build_prediction_prompt(
        self,
        row: pd.Series,
        target_id: str,
        examples: list[dict[str, Any]],
        cb_probs: dict[str, float],
    ) -> str:
        labels = labels_for[target_id]
        features_for_prompt = (
            self.available_prompt_features(
                row,
                target_id,
            )
        )

        labels_block = "\n".join(
            f"- {label}"
            for label in labels
        )
        probability_template = ",\n    ".join(
            f'"{label}": 0.0'
            for label in labels
        )
        prior_text = json.dumps(
            {
                label: round(
                    cb_probs[label],
                    4,
                )
                for label in labels
            },
            ensure_ascii=False,
        )

        label_note = LABEL_NOTES.get(
            target_id,
            "Labels are listed in their official scale order.",
        )

        return f"""
Target-specific reasoning guide:
{TARGET_GUIDES[target_id]}

Most important available respondent answers:
{render_evidence(
    row,
    features_for_prompt,
    max_items=self.config.prompt_features,
)}

Two compact labelled analogues from fit data:
{self._render_examples(examples, target_id)}

Auxiliary CatBoost probability prior:
{prior_text}

Target question:
{question_for[target_id]}

Allowed answer labels:
{labels_block}

Label interpretation:
{label_note}

Return JSON only:
{{
  "prediction": "one exact allowed label",
  "probabilities": {{
    {probability_template}
  }}
}}

Requirements:
- Include every allowed label exactly once.
- Probabilities must be numeric, non-negative and sum approximately to 1.
- The prediction must be the label with the largest probability.
- Do not include explanation, markdown or survey codes.
""".strip()

    def build_repair_prompt(
        self,
        target_id: str,
        raw_output: str,
    ) -> str:
        labels = labels_for[target_id]
        labels_block = "\n".join(
            f"- {label}"
            for label in labels
        )
        probability_template = ",\n    ".join(
            f'"{label}": 0.0'
            for label in labels
        )

        return f"""
Reformat the model output below into valid JSON.
Do not reconsider the survey case; preserve its intended classification
when it can be inferred.

Allowed labels:
{labels_block}

Original output:
{raw_output[:2500]}

Return exactly:
{{
  "prediction": "one exact allowed label",
  "probabilities": {{
    {probability_template}
  }}
}}
""".strip()

    def predict_one(
        self,
        row: pd.Series,
        target_id: str,
    ) -> dict[str, Any]:
        row_mode = self.row_mode(row)
        examples = self.retrievers[
            target_id
        ].retrieve(
            row,
            row_mode=row_mode,
        )
        cb_probs = self._catboost_probs(
            row,
            target_id,
        )

        prompt = self.build_prediction_prompt(
            row=row,
            target_id=target_id,
            examples=examples,
            cb_probs=cb_probs,
        )

        raw_first = ""
        raw_repair = ""
        parsed = None
        status = "success_first_pass"
        repair_used = False

        try:
            raw_first = self.llm.complete(
                PREDICTION_SYSTEM,
                prompt,
                thinking=False,
                max_tokens=self.config.first_pass_max_tokens,
            )
            parsed = parse_probability_response(
                raw_first,
                labels_for[target_id],
            )
        except Exception as error:
            raw_first = repr(error)

        if parsed is None:
            repair_used = True
            status = "repair_attempted"
            try:
                raw_repair = self.llm.complete(
                    REPAIR_SYSTEM,
                    self.build_repair_prompt(
                        target_id,
                        raw_first,
                    ),
                    thinking=False,
                    max_tokens=self.config.repair_max_tokens,
                )
                parsed = parse_probability_response(
                    raw_repair,
                    labels_for[target_id],
                )
                if parsed is not None:
                    status = "success_repair"
            except Exception as error:
                raw_repair = repr(error)

        if parsed is None:
            parsed = {
                "prediction": max(
                    cb_probs,
                    key=cb_probs.get,
                ),
                "probabilities": cb_probs,
            }
            status = "catboost_fallback"

        llm_probs = dict(
            parsed["probabilities"]
        )
        blend_weight = (
            self.config.catboost_blend_weight
        )

        final_probs = {
            label: (
                (1.0 - blend_weight)
                * llm_probs.get(label, 0.0)
                + blend_weight
                * cb_probs.get(label, 0.0)
            )
            for label in labels_for[target_id]
        }

        total = sum(final_probs.values())
        final_probs = {
            label: value / total
            for label, value in final_probs.items()
        }

        prediction = max(
            final_probs,
            key=final_probs.get,
        )

        return {
            "respondent_id": row.get("respondent_id"),
            "country": row.get("country"),
            "question_id": target_id,
            "prediction": prediction,
            "probabilities": final_probs,
            "llm_prediction": parsed["prediction"],
            "catboost_prediction": max(
                cb_probs,
                key=cb_probs.get,
            ),
            "row_mode": row_mode,
            "status": status,
            "repair_used": repair_used,
            "second_pass_used": False,
            "margin": probability_margin(
                final_probs
            ),
            "retrieved_ids": [
                example["respondent_id"]
                for example in examples
            ],
            "prompt_features": (
                self.available_prompt_features(
                    row,
                    target_id,
                )
            ),
            "raw_first": raw_first,
            "raw_repair": raw_repair,
        }

    def preview_prompt(
        self,
        row: pd.Series,
        target_id: str,
    ) -> str:
        examples = self.retrievers[
            target_id
        ].retrieve(
            row,
            row_mode=self.row_mode(row),
        )
        cb_probs = self._catboost_probs(
            row,
            target_id,
        )
        return self.build_prediction_prompt(
            row,
            target_id,
            examples,
            cb_probs,
        )



def audit_strict_feature_compliance(
    pipeline: FastNineCallLLMPipeline,
    sample_row: pd.Series,
    test_df: pd.DataFrame,
) -> pd.DataFrame:
    """
    Fail loudly if any predictive component uses a variable outside
    the official features dataset.
    """
    test_columns = set(
        test_df.columns.astype(str)
    )

    component_features = {
        "hybrid_selection": {
            feature
            for variables
            in pipeline.hybrid_selected.values()
            for feature in variables
        },
        "catboost": {
            feature
            for variables
            in pipeline.expert.features.values()
            for feature in variables
        },
        "retrieval": {
            feature
            for retriever
            in pipeline.retrievers.values()
            for feature in retriever.variables
        },
    }

    rows = []

    for component, used in component_features.items():
        outside_official_pool = (
            used - OFFICIAL_FEATURE_POOL
        )
        absent_from_test = (
            used - test_columns
        )
        target_leakage = (
            used & target_id_set
        )
        forbidden_identifiers = (
            used
            & {"country", "respondent_id"}
        )

        passed = (
            not outside_official_pool
            and not absent_from_test
            and not target_leakage
            and not forbidden_identifiers
        )

        rows.append({
            "component": component,
            "number_used": len(used),
            "outside_official_feature_pool": sorted(
                outside_official_pool
            ),
            "absent_from_test": sorted(
                absent_from_test
            ),
            "target_leakage": sorted(
                target_leakage
            ),
            "forbidden_identifiers": sorted(
                forbidden_identifiers
            ),
            "passed": passed,
        })

    audit = pd.DataFrame(rows)

    all_used = set().union(
        *component_features.values()
    )

    assert all_used <= OFFICIAL_FEATURE_POOL
    assert all_used <= test_columns
    assert not (all_used & target_id_set)
    assert "country" not in all_used
    assert "respondent_id" not in all_used

    for target_id in target_ids:
        prompt = pipeline.preview_prompt(
            sample_row,
            target_id,
        )

        assert (
            "Respondent country" not in prompt
        )
        assert (
            "respondent_id" not in prompt.lower()
        )

    print(
        "PASSED: model inputs are restricted to official feature variables. "
        "Country is used only for split construction and diagnostics."
    )

    return audit


## Block 17. Parallel runner with cache and checkpoints

One job equals one target-specific LLM call. For 1,050 test respondents, the pipeline creates 9,450 primary jobs.


In [17]:
def run_fast9_predictions(
    pipeline: FastNineCallLLMPipeline,
    input_df: pd.DataFrame,
    run_id: str,
    include_truth: bool,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    truth_df = input_df.copy().reset_index(drop=True)
    masked_df = mask_target_columns(
        truth_df
    ).reset_index(drop=True)

    checkpoint = Path(
        f"fast9_{run_id}_checkpoint.jsonl"
    )
    completed = {}

    if checkpoint.exists():
        with checkpoint.open(
            "r",
            encoding="utf-8",
        ) as file:
            for line in file:
                try:
                    record = json.loads(line)
                    key = (
                        str(record["respondent_id"]),
                        record["question_id"],
                    )
                    completed[key] = record
                except Exception:
                    pass

    truth_lookup = {}
    jobs = []

    for index, row in masked_df.iterrows():
        truth_row = truth_df.iloc[index]
        respondent_id = str(
            row["respondent_id"]
        )

        for target_id in target_ids:
            key = (
                respondent_id,
                target_id,
            )

            if include_truth:
                truth = decode_target_value(
                    target_id,
                    truth_row[target_id],
                )
                if truth is None:
                    continue
                truth_lookup[key] = truth

            if key not in completed:
                jobs.append((
                    row,
                    target_id,
                ))

    print(
        f"New LLM jobs: {len(jobs):,}; "
        f"already checkpointed: {len(completed):,}"
    )

    buffer = []
    buffer_lock = threading.Lock()
    started = time.perf_counter()

    def task(
        row: pd.Series,
        target_id: str,
    ):
        record = pipeline.predict_one(
            row,
            target_id,
        )
        key = (
            str(row["respondent_id"]),
            target_id,
        )
        if include_truth:
            record["truth"] = (
                truth_lookup[key]
            )
        record["probabilities"] = {
            label: float(value)
            for label, value in record[
                "probabilities"
            ].items()
        }
        return record

    def flush_buffer():
        nonlocal buffer
        if not buffer:
            return
        with checkpoint.open(
            "a",
            encoding="utf-8",
        ) as file:
            for item in buffer:
                file.write(
                    json.dumps(
                        item,
                        ensure_ascii=False,
                    )
                    + "\n"
                )
        buffer = []

    with ThreadPoolExecutor(
        max_workers=pipeline.config.max_workers
    ) as pool:
        future_map = {
            pool.submit(
                task,
                row,
                target_id,
            ): (
                str(row["respondent_id"]),
                target_id,
            )
            for row, target_id in jobs
        }

        for future in tqdm(
            as_completed(future_map),
            total=len(future_map),
            desc=run_id,
        ):
            key = future_map[future]

            try:
                record = future.result()
            except Exception as error:
                respondent_id, target_id = key
                row = masked_df[
                    masked_df["respondent_id"]
                    .astype(str)
                    .eq(respondent_id)
                ].iloc[0]

                cb_probs = pipeline._catboost_probs(
                    row,
                    target_id,
                )
                record = {
                    "respondent_id": row[
                        "respondent_id"
                    ],
                    "country": row["country"],
                    "question_id": target_id,
                    "prediction": max(
                        cb_probs,
                        key=cb_probs.get,
                    ),
                    "probabilities": cb_probs,
                    "llm_prediction": None,
                    "catboost_prediction": max(
                        cb_probs,
                        key=cb_probs.get,
                    ),
                    "row_mode": pipeline.row_mode(
                        row
                    ),
                    "status": (
                        "unhandled_error_"
                        "catboost_fallback"
                    ),
                    "repair_used": False,
                    "second_pass_used": False,
                    "margin": probability_margin(
                        cb_probs
                    ),
                    "retrieved_ids": [],
                    "prompt_features": [],
                    "raw_first": repr(error),
                    "raw_repair": "",
                }
                if include_truth:
                    record["truth"] = (
                        truth_lookup[key]
                    )

            completed[key] = record
            buffer.append(record)

            if (
                len(buffer)
                >= pipeline.config.checkpoint_every
            ):
                with buffer_lock:
                    flush_buffer()

    with buffer_lock:
        flush_buffer()

    # Rewrite a clean, deduplicated checkpoint.
    with checkpoint.open(
        "w",
        encoding="utf-8",
    ) as file:
        for key in sorted(completed):
            file.write(
                json.dumps(
                    completed[key],
                    ensure_ascii=False,
                )
                + "\n"
            )

    raw = pd.DataFrame(
        completed.values()
    ).drop_duplicates(
        ["respondent_id", "question_id"],
        keep="last",
    )

    wanted_ids = set(
        truth_df["respondent_id"].astype(str)
    )
    raw = raw[
        raw["respondent_id"]
        .astype(str)
        .isin(wanted_ids)
    ].copy()

    predictions = raw[
        [
            "respondent_id",
            "question_id",
            "prediction",
        ]
    ].copy()

    expected = (
        len(truth_lookup)
        if include_truth
        else len(truth_df) * len(target_ids)
    )

    assert len(raw) == expected, (
        len(raw),
        expected,
    )
    assert not raw.duplicated(
        ["respondent_id", "question_id"]
    ).any()

    for target_id, group in raw.groupby(
        "question_id"
    ):
        assert set(
            group["prediction"]
        ).issubset(
            set(labels_for[target_id])
        )

    elapsed_minutes = (
        time.perf_counter() - started
    ) / 60

    print(
        f"Completed {len(raw):,} predictions "
        f"in {elapsed_minutes:.2f} minutes."
    )
    print(
        "Repair share:",
        f"{raw['repair_used'].mean():.2%}",
    )
    print(
        "Fallback share:",
        f"{raw['status'].str.contains('fallback').mean():.2%}",
    )

    return (
        predictions.reset_index(drop=True),
        raw.reset_index(drop=True),
    )


## Block 18. Scoring and paired diagnostics

In [18]:
def balanced_respondent_sample(df: pd.DataFrame, n: Optional[int], seed: int) -> pd.DataFrame:
    if n is None or n >= len(df):
        return df.copy().reset_index(drop=True)
    rng = np.random.default_rng(seed)
    countries = list(df["country"].dropna().unique())
    selected = []
    country_indices = {
        c: rng.permutation(df.index[df["country"] == c].to_numpy()).tolist()
        for c in countries
    }
    while len(selected) < n:
        progress = False
        for c in countries:
            if country_indices[c] and len(selected) < n:
                selected.append(country_indices[c].pop())
                progress = True
        if not progress:
            break
    return df.loc[selected].copy().reset_index(drop=True)


def build_profiles_batch(
    pipeline: AlternativeLLMPipeline,
    masked_df: pd.DataFrame,
    workers: Optional[int] = None,
) -> dict[str, str]:
    if not pipeline.config.use_profile_stage:
        return {str(row["respondent_id"]): "Profile stage disabled." for _, row in masked_df.iterrows()}

    workers = workers or pipeline.config.max_workers
    profiles = {}
    with ThreadPoolExecutor(max_workers=workers) as pool:
        future_map = {
            pool.submit(pipeline.get_profile, row): str(row["respondent_id"])
            for _, row in masked_df.iterrows()
        }
        for future in tqdm(as_completed(future_map), total=len(future_map), desc="Profiles"):
            rid = future_map[future]
            try:
                profiles[rid] = future.result()
            except Exception as exc:
                profiles[rid] = f"Profile unavailable: {exc!r}"
    return profiles


def run_local_validation(
    pipeline: AlternativeLLMPipeline,
    valid_df: pd.DataFrame,
    experiment_name: str,
    max_eval_respondents: Optional[int] = 40,
) -> pd.DataFrame:
    eval_truth = balanced_respondent_sample(valid_df, max_eval_respondents, pipeline.config.seed)
    eval_masked = mask_target_columns(eval_truth)
    profiles = build_profiles_batch(pipeline, eval_masked)

    checkpoint = Path(f"validation_{experiment_name}_checkpoint.jsonl")
    completed = {}
    if checkpoint.exists():
        with checkpoint.open("r", encoding="utf-8") as f:
            for line in f:
                try:
                    rec = json.loads(line)
                    completed[(str(rec["respondent_id"]), rec["question_id"])] = rec
                except Exception:
                    pass

    jobs = []
    truth_lookup = {}
    for idx, row in eval_masked.iterrows():
        truth_row = eval_truth.iloc[idx]
        rid = str(row["respondent_id"])
        for target_id in target_ids:
            truth = decode_target_value(target_id, truth_row[target_id])
            if truth is None:
                continue
            truth_lookup[(rid, target_id)] = truth
            if (rid, target_id) not in completed:
                jobs.append((row, target_id, profiles[rid]))

    new_records = []
    lock = threading.Lock()

    def task(row, target_id, profile):
        rec = pipeline.predict_one(row, target_id, profile=profile)
        rec["truth"] = truth_lookup[(str(row["respondent_id"]), target_id)]
        rec["probabilities"] = {k: float(v) for k, v in rec["probabilities"].items()}
        return rec

    with ThreadPoolExecutor(max_workers=pipeline.config.max_workers) as pool:
        futures = [pool.submit(task, *job) for job in jobs]
        for future in tqdm(as_completed(futures), total=len(futures), desc=experiment_name):
            rec = future.result()
            new_records.append(rec)
            completed[(str(rec["respondent_id"]), rec["question_id"])] = rec
            if len(new_records) % pipeline.config.checkpoint_every == 0:
                with lock, checkpoint.open("a", encoding="utf-8") as f:
                    for item in new_records[-pipeline.config.checkpoint_every:]:
                        f.write(json.dumps(item, ensure_ascii=False) + "\n")

    with checkpoint.open("w", encoding="utf-8") as f:
        for rec in completed.values():
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")

    result = pd.DataFrame(completed.values())
    expected_ids = set(eval_truth["respondent_id"].astype(str))
    result = result[result["respondent_id"].astype(str).isin(expected_ids)].copy()
    return result.reset_index(drop=True)


def score_predictions(predictions: pd.DataFrame, majority: dict[str, str], prediction_col: str = "prediction"):
    rows = []
    for target_id in target_ids:
        g = predictions[predictions["question_id"] == target_id].copy()
        if g.empty:
            continue
        valid_mask = g[prediction_col].isin(labels_for[target_id])
        coverage = float(valid_mask.mean())
        y_true = g["truth"].astype(str)
        y_pred = g[prediction_col].fillna("__INVALID__").astype(str)

        accuracy = float(accuracy_score(y_true, y_pred))
        macro_f1 = float(f1_score(
            y_true,
            y_pred,
            labels=labels_for[target_id],
            average="macro",
            zero_division=0,
        ))
        majority_accuracy = float((y_true == majority[target_id]).mean())
        skill_proxy = (
            (accuracy - majority_accuracy) / (1 - majority_accuracy)
            if majority_accuracy < 1 else 0.0
        )

        true_dist = y_true.value_counts(normalize=True).reindex(labels_for[target_id], fill_value=0.0)
        pred_dist = y_pred.value_counts(normalize=True).reindex(labels_for[target_id], fill_value=0.0)
        alignment_proxy = float(1 - 0.5 * np.abs(true_dist - pred_dist).sum())

        rows.append({
            "question_id": target_id,
            "n": len(g),
            "coverage": coverage,
            "accuracy": accuracy,
            "macro_f1": macro_f1,
            "majority_accuracy": majority_accuracy,
            "skill_proxy": skill_proxy,
            "alignment_proxy": alignment_proxy,
        })

    per_target = pd.DataFrame(rows)
    overall = pd.DataFrame([{
        "question_id": "MEAN_OVER_TARGETS",
        "n": int(per_target["n"].sum()),
        "coverage": per_target["coverage"].mean(),
        "accuracy": per_target["accuracy"].mean(),
        "macro_f1": per_target["macro_f1"].mean(),
        "majority_accuracy": per_target["majority_accuracy"].mean(),
        "skill_proxy": per_target["skill_proxy"].mean(),
        "alignment_proxy": per_target["alignment_proxy"].mean(),
    }])
    return pd.concat([per_target, overall], ignore_index=True)


def majority_baseline_frame(valid_df: pd.DataFrame, majority: dict[str, str], max_n: Optional[int], seed: int):
    sampled = balanced_respondent_sample(valid_df, max_n, seed)
    records = []
    for _, row in sampled.iterrows():
        for target_id in target_ids:
            truth = decode_target_value(target_id, row[target_id])
            if truth is None:
                continue
            records.append({
                "respondent_id": row["respondent_id"],
                "question_id": target_id,
                "truth": truth,
                "prediction": majority[target_id],
            })
    return pd.DataFrame(records)


def config_signature(config: PipelineConfig, prefix: str) -> str:
    payload = asdict(config)
    payload.pop("cache_path", None)
    text = json.dumps(
        {"prefix": prefix, "config": payload},
        ensure_ascii=False,
        sort_keys=True,
        default=str,
    )
    return hashlib.sha256(text.encode("utf-8")).hexdigest()[:10]


def make_fixed_comparison_sample(
    valid_df: pd.DataFrame,
    n_respondents: Optional[int],
    seed: int,
) -> pd.DataFrame:
    """Choose respondents once, then pass exactly this frame to both methods."""
    sample = balanced_respondent_sample(
        valid_df,
        n=n_respondents,
        seed=seed,
    )
    assert sample["respondent_id"].is_unique
    return sample.reset_index(drop=True)


def paired_prediction_frame(
    legacy_result: pd.DataFrame,
    alternative_result: pd.DataFrame,
) -> pd.DataFrame:
    keys = ["respondent_id", "question_id"]

    old = legacy_result[
        keys + ["country", "truth", "prediction", "status"]
    ].rename(columns={
        "prediction": "legacy_prediction",
        "status": "legacy_status",
    })

    new = alternative_result[
        keys + ["truth", "prediction", "status", "second_pass_used"]
    ].rename(columns={
        "truth": "alternative_truth",
        "prediction": "alternative_prediction",
        "status": "alternative_status",
    })

    paired = old.merge(
        new,
        on=keys,
        how="inner",
        validate="one_to_one",
    )

    if len(paired) != len(old) or len(paired) != len(new):
        raise AssertionError(
            "The two methods were not evaluated on exactly the same "
            "respondent-target pairs."
        )

    if not (
        paired["truth"].astype(str)
        == paired["alternative_truth"].astype(str)
    ).all():
        raise AssertionError("Truth labels differ between methods.")

    paired = paired.drop(columns="alternative_truth")
    paired["legacy_correct"] = (
        paired["legacy_prediction"] == paired["truth"]
    )
    paired["alternative_correct"] = (
        paired["alternative_prediction"] == paired["truth"]
    )
    paired["alternative_win"] = (
        paired["alternative_correct"]
        & ~paired["legacy_correct"]
    )
    paired["alternative_loss"] = (
        paired["legacy_correct"]
        & ~paired["alternative_correct"]
    )
    return paired


def compare_score_tables(
    legacy_result: pd.DataFrame,
    alternative_result: pd.DataFrame,
    legacy_majority: dict[str, str],
    alternative_majority: dict[str, str],
) -> pd.DataFrame:
    old_score = score_predictions(
        legacy_result,
        legacy_majority,
    ).rename(columns={
        column: f"legacy_{column}"
        for column in [
            "n",
            "coverage",
            "accuracy",
            "macro_f1",
            "majority_accuracy",
            "skill_proxy",
            "alignment_proxy",
        ]
    })

    new_score = score_predictions(
        alternative_result,
        alternative_majority,
    ).rename(columns={
        column: f"alternative_{column}"
        for column in [
            "n",
            "coverage",
            "accuracy",
            "macro_f1",
            "majority_accuracy",
            "skill_proxy",
            "alignment_proxy",
        ]
    })

    comparison = old_score.merge(
        new_score,
        on="question_id",
        how="inner",
        validate="one_to_one",
    )

    for metric in [
        "coverage",
        "accuracy",
        "macro_f1",
        "skill_proxy",
        "alignment_proxy",
    ]:
        comparison[f"delta_{metric}"] = (
            comparison[f"alternative_{metric}"]
            - comparison[f"legacy_{metric}"]
        )

    return comparison


def _mean_target_macro_f1(
    frame: pd.DataFrame,
    prediction_col: str,
) -> float:
    scores = []
    for target_id in target_ids:
        group = frame[frame["question_id"] == target_id]
        if group.empty:
            continue
        scores.append(
            f1_score(
                group["truth"].astype(str),
                group[prediction_col].astype(str),
                labels=labels_for[target_id],
                average="macro",
                zero_division=0,
            )
        )
    return float(np.mean(scores)) if scores else float("nan")


def _mean_target_alignment(
    frame: pd.DataFrame,
    prediction_col: str,
) -> float:
    values = []
    for target_id in target_ids:
        group = frame[frame["question_id"] == target_id]
        if group.empty:
            continue
        labels = labels_for[target_id]
        truth_dist = (
            group["truth"]
            .astype(str)
            .value_counts(normalize=True)
            .reindex(labels, fill_value=0.0)
        )
        pred_dist = (
            group[prediction_col]
            .astype(str)
            .value_counts(normalize=True)
            .reindex(labels, fill_value=0.0)
        )
        values.append(
            1.0 - 0.5 * np.abs(truth_dist - pred_dist).sum()
        )
    return float(np.mean(values)) if values else float("nan")


def paired_cluster_bootstrap(
    paired: pd.DataFrame,
    cluster_col: str = "country",
    n_bootstrap: int = 1000,
    seed: int = 42,
) -> pd.DataFrame:
    """
    Bootstrap deltas while preserving all rows inside a country cluster.

    The reported alignment is the notebook's local distribution-alignment
    proxy, not necessarily the hidden leaderboard implementation.
    """
    required = {
        cluster_col,
        "question_id",
        "truth",
        "legacy_prediction",
        "alternative_prediction",
    }
    missing = required - set(paired.columns)
    if missing:
        raise ValueError(f"Missing columns: {sorted(missing)}")

    clusters = paired[cluster_col].dropna().astype(str).unique()
    if len(clusters) < 2:
        raise ValueError("At least two bootstrap clusters are required.")

    rng = np.random.default_rng(seed)
    boot_rows = []

    for _ in range(n_bootstrap):
        sampled_clusters = rng.choice(
            clusters,
            size=len(clusters),
            replace=True,
        )
        sampled = pd.concat(
            [
                paired[
                    paired[cluster_col].astype(str) == cluster
                ]
                for cluster in sampled_clusters
            ],
            ignore_index=True,
        )

        legacy_f1 = _mean_target_macro_f1(
            sampled,
            "legacy_prediction",
        )
        alternative_f1 = _mean_target_macro_f1(
            sampled,
            "alternative_prediction",
        )

        legacy_alignment = _mean_target_alignment(
            sampled,
            "legacy_prediction",
        )
        alternative_alignment = _mean_target_alignment(
            sampled,
            "alternative_prediction",
        )

        legacy_accuracy = float(
            (
                sampled["legacy_prediction"]
                == sampled["truth"]
            ).mean()
        )
        alternative_accuracy = float(
            (
                sampled["alternative_prediction"]
                == sampled["truth"]
            ).mean()
        )

        boot_rows.append({
            "delta_macro_f1": alternative_f1 - legacy_f1,
            "delta_alignment_proxy": (
                alternative_alignment - legacy_alignment
            ),
            "delta_accuracy": (
                alternative_accuracy - legacy_accuracy
            ),
        })

    boot = pd.DataFrame(boot_rows)

    observed = {
        "delta_macro_f1": (
            _mean_target_macro_f1(
                paired,
                "alternative_prediction",
            )
            - _mean_target_macro_f1(
                paired,
                "legacy_prediction",
            )
        ),
        "delta_alignment_proxy": (
            _mean_target_alignment(
                paired,
                "alternative_prediction",
            )
            - _mean_target_alignment(
                paired,
                "legacy_prediction",
            )
        ),
        "delta_accuracy": float(
            (
                paired["alternative_prediction"]
                == paired["truth"]
            ).mean()
            - (
                paired["legacy_prediction"]
                == paired["truth"]
            ).mean()
        ),
    }

    summary = []
    for metric, estimate in observed.items():
        values = boot[metric].to_numpy()
        summary.append({
            "metric": metric,
            "observed_delta": estimate,
            "ci_2_5": float(np.quantile(values, 0.025)),
            "ci_97_5": float(np.quantile(values, 0.975)),
            "bootstrap_probability_delta_gt_0": float(
                np.mean(values > 0)
            ),
        })

    return pd.DataFrame(summary)


def paired_win_loss_table(
    paired: pd.DataFrame,
) -> pd.DataFrame:
    rows = []
    for target_id in target_ids + ["ALL_TARGETS"]:
        group = (
            paired
            if target_id == "ALL_TARGETS"
            else paired[paired["question_id"] == target_id]
        )
        if group.empty:
            continue

        wins = int(group["alternative_win"].sum())
        losses = int(group["alternative_loss"].sum())
        ties_both_correct = int(
            (
                group["legacy_correct"]
                & group["alternative_correct"]
            ).sum()
        )
        ties_both_wrong = int(
            (
                ~group["legacy_correct"]
                & ~group["alternative_correct"]
            ).sum()
        )

        rows.append({
            "question_id": target_id,
            "n": len(group),
            "alternative_wins": wins,
            "alternative_losses": losses,
            "net_wins": wins - losses,
            "both_correct": ties_both_correct,
            "both_wrong": ties_both_wrong,
        })

    return pd.DataFrame(rows)


def three_way_score_table(
    legacy_result: pd.DataFrame,
    full_alternative_result: pd.DataFrame,
    fast9_result: pd.DataFrame,
    majority: dict[str, str],
) -> pd.DataFrame:
    frames = []

    for method_name, result in [
        ("legacy", legacy_result),
        ("full_alternative", full_alternative_result),
        ("fast9", fast9_result),
    ]:
        score = score_predictions(
            result,
            majority,
        )
        score = score.rename(columns={
            column: f"{method_name}_{column}"
            for column in [
                "n",
                "coverage",
                "accuracy",
                "macro_f1",
                "majority_accuracy",
                "skill_proxy",
                "alignment_proxy",
            ]
        })
        frames.append(score)

    combined = frames[0]
    for frame in frames[1:]:
        combined = combined.merge(
            frame,
            on="question_id",
            how="inner",
            validate="one_to_one",
        )

    for metric in [
        "coverage",
        "accuracy",
        "macro_f1",
        "skill_proxy",
        "alignment_proxy",
    ]:
        combined[
            f"fast9_minus_legacy_{metric}"
        ] = (
            combined[f"fast9_{metric}"]
            - combined[f"legacy_{metric}"]
        )
        combined[
            f"fast9_minus_full_{metric}"
        ] = (
            combined[f"fast9_{metric}"]
            - combined[
                f"full_alternative_{metric}"
            ]
        )

    return combined


def gain_retention(
    legacy_value: float,
    full_value: float,
    fast_value: float,
) -> float:
    denominator = full_value - legacy_value
    if denominator <= 0:
        return float("nan")
    return (
        fast_value - legacy_value
    ) / denominator


# Part II. Fast validation using the saved OOD pilot

There is no need to rerun the first pipeline or the full alternative pipeline.

## Block 19. Upload `pilot_ood_results_*.zip`


In [19]:
from google.colab import files

uploaded = files.upload()

zip_names = [
    name
    for name in uploaded
    if name.lower().endswith(".zip")
]

if not zip_names:
    raise ValueError(
        "Upload pilot_ood_results_*.zip"
    )

REFERENCE_DIR = Path(
    "/content/fast9_reference"
)
shutil.rmtree(
    REFERENCE_DIR,
    ignore_errors=True,
)
REFERENCE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

with zipfile.ZipFile(
    zip_names[0],
    "r",
) as archive:
    archive.extractall(
        REFERENCE_DIR
    )

def find_required_file(
    filename: str,
) -> Path:
    matches = list(
        REFERENCE_DIR.rglob(filename)
    )
    if not matches:
        raise FileNotFoundError(
            f"{filename} was not found "
            f"in {zip_names[0]}"
        )
    return matches[0]


reference_sample = pd.read_csv(
    find_required_file("sample.csv")
)
reference_legacy = pd.read_csv(
    find_required_file("legacy_result.csv")
)
reference_full = pd.read_csv(
    find_required_file("alternative_result.csv")
)

for frame in [
    reference_sample,
    reference_legacy,
    reference_full,
]:
    frame["respondent_id"] = (
        frame["respondent_id"].astype(str)
    )

print(
    "Reference respondents:",
    reference_sample[
        "respondent_id"
    ].nunique(),
)
print(
    "Legacy pairs:",
    len(reference_legacy),
)
print(
    "Full alternative pairs:",
    len(reference_full),
)

assert set(
    reference_legacy[
        ["respondent_id", "question_id"]
    ].itertuples(index=False, name=None)
) == set(
    reference_full[
        ["respondent_id", "question_id"]
    ].itertuples(index=False, name=None)
)


Saving pilot_ood_results_20260714_213132.zip to pilot_ood_results_20260714_213132.zip
Reference respondents: 75
Legacy pairs: 650
Full alternative pairs: 650


## Block 20. Fit Fast9 on the same leakage-safe OOD fit fold

This cell does not call the LLM.


In [20]:
ood_split = make_out_domain_split(
    train,
    n_countries=CFG.out_domain_n_countries,
    val_per_country=CFG.out_domain_val_per_country,
    seed=CFG.seed,
)

reference_ids = set(
    reference_sample[
        "respondent_id"
    ].astype(str)
)
valid_ids = set(
    ood_split.valid[
        "respondent_id"
    ].astype(str)
)

missing_reference_ids = (
    reference_ids - valid_ids
)
if missing_reference_ids:
    raise AssertionError(
        "The reconstructed split does not "
        "match the saved pilot sample. "
        f"Missing IDs: "
        f"{list(missing_reference_ids)[:5]}"
    )

validation_cfg = copy.deepcopy(
    CFG
)
validation_cfg.cache_path = (
    "llm_cache_fast9_strict_features_validation.jsonl"
)

validation_llm = CachedNebiusClient(
    validation_cfg,
    api_key=nebius_key,
    client=raw_nebius_client,
)

fast9_validation_pipeline = (
    FastNineCallLLMPipeline(
        validation_cfg,
        validation_llm,
    )
    .fit(
        ood_split.fit,
        mode=ood_split.mode,
    )
)

strict_validation_feature_audit = (
    audit_strict_feature_compliance(
        pipeline=fast9_validation_pipeline,
        sample_row=mask_target_columns(
            reference_sample
        ).iloc[0],
        test_df=test,
    )
)

display(
    strict_validation_feature_audit
)

display(
    fast9_validation_pipeline
    .feature_importance_table
    .query(
        "selected_for_prompt_or_retrieval"
    )
    .groupby("question_id")
    .head(16)[
        [
            "question_id",
            "feature",
            "question",
            "protected_mrmr",
            "catboost_top_addition",
            "adjusted_score",
            "catboost_importance",
            "hybrid_score",
        ]
    ]
)


Loaded 0 cached LLM responses


Encoding feature columns:   0%|          | 0/278 [00:00<?, ?it/s]

Target feature selection:   0%|          | 0/9 [00:00<?, ?it/s]

Pipeline fit completed in 1.05 minutes.
PASSED: model inputs are restricted to official feature variables. Country is used only for split construction and diagnostics.


,component,number_used,outside_official_feature_pool,absent_from_test,target_leakage,forbidden_identifiers,passed
0,hybrid_selection,84,[],[],[],[],True
1,catboost,105,[],[],[],[],True
2,retrieval,84,[],[],[],[],True


,question_id,feature,question,protected_mrmr,catboost_top_addition,adjusted_score,catboost_importance,hybrid_score
0,Q201,Q203,How often do you use radio news to obtain information about this country and the world?,True,True,0.073650,1.000000,0.935882
1,Q201,Q223,"If there were a national election tomorrow, for which party would you vote? If you do not know, which party appeals ...",True,True,0.081028,0.539691,0.882578
2,Q201,N_REGION_ISO,Which region do you currently live in?,False,False,0.079545,0.000000,0.784106
3,Q201,Q202,How often do you use tv news to obtain information about this country and the world?,True,True,0.057734,0.617523,0.722152
4,Q201,Q290,How do you identify when it comes to Racial belonging/ ethnic group in your country?,False,False,0.067510,0.000000,0.668043
5,Q201,Q268,In which country was your father born? List of codes in Annex,False,False,0.062825,0.000000,0.632063
6,Q201,Q267,In which country was your mother born?,False,False,0.062681,0.000000,0.630961
7,Q201,Q266,In which country were you born?,False,False,0.061474,0.000000,0.627932
8,Q201,Q272,What language do you normally speak at home?,False,False,0.060843,0.000000,0.622513
9,Q201,Q205,How often do you use email to obtain information about this country and the world?,True,True,0.046991,0.504285,0.605829


## Блок 21. Preview strict prompt до новых API-вызовов

В prompt не должно быть отдельного блока с `country`. Названия стран могут встречаться только как декодированные ответы на официально разрешённые survey features, например на вопрос о стране рождения.

In [21]:
preview_row = mask_target_columns(
    reference_sample
).iloc[0]

strict_preview_prompt = (
    fast9_validation_pipeline.preview_prompt(
        preview_row,
        "Q17",
    )
)

assert (
    "Respondent country" not in strict_preview_prompt
)
assert (
    "respondent_id"
    not in strict_preview_prompt.lower()
)

print(
    strict_preview_prompt
)


Target-specific reasoning guide:
Infer preference for child obedience versus autonomy. Consider religion, authority orientation, family values, traditionalism and whether independence is valued for children.

Most important available respondent answers:
- [Q290] How do you identify when it comes to Racial belonging/ ethnic group in your country?
  Answer: BO: Not pertaining to Indigenous groups
- [Q8] Do you consider independence to be an especially important quality for children to learn at home?
  Answer: Not so important
- [Q45] If society placed more emphasis on developing technology in the future, would you consider that a good thing, a bad thing, or wouldn't you mind?
  Answer: Good thing
- [Q57] Generally speaking, would you say that most people can be trusted or that you need to be very careful in dealing with people?
  Answer: Need to be very careful
- [Q11] Do you consider imagination to be an especially important quality for children to learn at home?
  Answer: Not so import

## Block 22. Run the new validation

For 75 respondents, this requires at most approximately 650–675 primary LLM calls, executed in parallel.


In [22]:
validation_started = time.perf_counter()

fast9_validation_predictions, fast9_validation_result = (
    run_fast9_predictions(
        pipeline=fast9_validation_pipeline,
        input_df=reference_sample,
        run_id="validation_ood_strict_features_seed42_n75",
        include_truth=True,
    )
)

validation_elapsed = (
    time.perf_counter()
    - validation_started
) / 60

print(
    f"Total validation wall time: "
    f"{validation_elapsed:.2f} minutes."
)


New LLM jobs: 650; already checkpointed: 0


validation_ood_strict_features_seed42_n75:   0%|          | 0/650 [00:00<?, ?it/s]

Completed 650 predictions in 0.70 minutes.
Repair share: 0.00%
Fallback share: 0.00%
Total validation wall time: 0.70 minutes.


## Block 23. Compare the three methods

Main questions:

- Does Fast9 outperform the first pipeline?
- What share of the full alternative pipeline's improvement does it retain?
- Are the improvements for Q17, Q186, and Q242 preserved?


In [23]:
score_comparison = three_way_score_table(
    legacy_result=reference_legacy,
    full_alternative_result=reference_full,
    fast9_result=fast9_validation_result,
    majority=fast9_validation_pipeline.majority,
)

display(
    score_comparison[
        [
            "question_id",
            "legacy_macro_f1",
            "full_alternative_macro_f1",
            "fast9_macro_f1",
            "fast9_minus_legacy_macro_f1",
            "fast9_minus_full_macro_f1",
            "legacy_accuracy",
            "full_alternative_accuracy",
            "fast9_accuracy",
            "legacy_alignment_proxy",
            "full_alternative_alignment_proxy",
            "fast9_alignment_proxy",
            "fast9_coverage",
        ]
    ]
)

overall = (
    score_comparison
    .query(
        "question_id == 'MEAN_OVER_TARGETS'"
    )
    .iloc[0]
)

retention_summary = pd.DataFrame([
    {
        "metric": metric,
        "legacy": float(
            overall[f"legacy_{metric}"]
        ),
        "full_alternative": float(
            overall[
                f"full_alternative_{metric}"
            ]
        ),
        "fast9": float(
            overall[f"fast9_{metric}"]
        ),
        "fast9_minus_legacy": float(
            overall[
                f"fast9_minus_legacy_{metric}"
            ]
        ),
        "retained_share_of_full_gain": (
            gain_retention(
                float(
                    overall[
                        f"legacy_{metric}"
                    ]
                ),
                float(
                    overall[
                        f"full_alternative_{metric}"
                    ]
                ),
                float(
                    overall[
                        f"fast9_{metric}"
                    ]
                ),
            )
        ),
    }
    for metric in [
        "macro_f1",
        "accuracy",
        "alignment_proxy",
    ]
])

display(retention_summary)


paired_vs_legacy = paired_prediction_frame(
    reference_legacy,
    fast9_validation_result,
)
bootstrap_vs_legacy = paired_cluster_bootstrap(
    paired_vs_legacy,
    cluster_col="country",
    n_bootstrap=1000,
    seed=CFG.seed,
)
win_loss_vs_legacy = paired_win_loss_table(
    paired_vs_legacy
)

paired_vs_full = paired_prediction_frame(
    reference_full,
    fast9_validation_result,
)
bootstrap_vs_full = paired_cluster_bootstrap(
    paired_vs_full,
    cluster_col="country",
    n_bootstrap=1000,
    seed=CFG.seed,
)
win_loss_vs_full = paired_win_loss_table(
    paired_vs_full
)

print("FAST9 versus first pipeline")
display(bootstrap_vs_legacy)
display(win_loss_vs_legacy)

print("FAST9 versus full alternative")
display(bootstrap_vs_full)
display(win_loss_vs_full)


,question_id,legacy_macro_f1,full_alternative_macro_f1,fast9_macro_f1,fast9_minus_legacy_macro_f1,fast9_minus_full_macro_f1,legacy_accuracy,full_alternative_accuracy,fast9_accuracy,legacy_alignment_proxy,full_alternative_alignment_proxy,fast9_alignment_proxy,fast9_coverage
0,Q201,0.321391,0.332393,0.276607,-0.044784,-0.055786,0.391892,0.391892,0.351351,0.729730,0.770270,0.851351,1.0
1,Q73,0.697922,0.768931,0.762500,0.064578,-0.006431,0.785714,0.828571,0.800000,0.885714,0.900000,0.914286,1.0
2,Q227,0.498685,0.526710,0.499310,0.000625,-0.027401,0.507246,0.536232,0.507246,0.869565,0.869565,0.840580,1.0
3,Q209,0.681455,0.654762,0.741898,0.060443,0.087136,0.690141,0.676056,0.760563,0.816901,0.802817,0.887324,1.0
4,Q33,0.390222,0.449841,0.438807,0.048585,-0.011034,0.465753,0.465753,0.452055,0.630137,0.753425,0.780822,1.0
5,Q148,0.668700,0.686720,0.748209,0.079509,0.061490,0.693333,0.706667,0.760000,0.866667,0.906667,0.920000,1.0
6,Q17,0.400000,0.638796,0.573591,0.173591,-0.065205,0.416667,0.666667,0.625000,0.527778,0.972222,0.958333,1.0
7,Q186,0.508362,0.674881,0.757978,0.249615,0.083096,0.589041,0.698630,0.767123,0.753425,0.876712,0.876712,1.0
8,Q242,0.140316,0.328432,0.312616,0.172299,-0.015817,0.328767,0.397260,0.424658,0.328767,0.767123,0.835616,1.0
9,MEAN_OVER_TARGETS,0.478561,0.562385,0.567946,0.089385,0.005561,0.540951,0.596414,0.605333,0.712076,0.846534,0.873892,1.0


,metric,legacy,full_alternative,fast9,fast9_minus_legacy,retained_share_of_full_gain
0,macro_f1,0.478561,0.562385,0.567946,0.089385,1.066342
1,accuracy,0.540951,0.596414,0.605333,0.064382,1.160802
2,alignment_proxy,0.712076,0.846534,0.873892,0.161816,1.203470


FAST9 versus first pipeline


,metric,observed_delta,ci_2_5,ci_97_5,bootstrap_probability_delta_gt_0
0,delta_macro_f1,0.089385,0.042176,0.126583,1.000
1,delta_alignment_proxy,0.161816,0.103494,0.197165,1.000
2,delta_accuracy,0.064615,0.010802,0.107830,0.989


,question_id,n,alternative_wins,alternative_losses,net_wins,both_correct,both_wrong
0,Q201,74,6,9,-3,20,39
1,Q73,70,4,3,1,52,11
2,Q227,69,3,3,0,32,31
3,Q209,71,7,2,5,47,15
4,Q33,73,7,8,-1,26,32
5,Q148,75,7,2,5,50,16
6,Q17,72,28,13,15,17,14
7,Q186,73,16,3,13,40,14
8,Q242,73,16,9,7,15,33
9,ALL_TARGETS,650,94,52,42,299,205


FAST9 versus full alternative


,metric,observed_delta,ci_2_5,ci_97_5,bootstrap_probability_delta_gt_0
0,delta_macro_f1,0.005561,-0.017136,0.034037,0.692
1,delta_alignment_proxy,0.027358,0.003885,0.063694,0.991
2,delta_accuracy,0.009231,-0.015291,0.037523,0.723


,question_id,n,alternative_wins,alternative_losses,net_wins,both_correct,both_wrong
0,Q201,74,3,6,-3,23,42
1,Q73,70,1,3,-2,55,11
2,Q227,69,0,2,-2,35,32
3,Q209,71,6,0,6,48,17
4,Q33,73,5,6,-1,28,34
5,Q148,75,4,0,4,53,18
6,Q17,72,9,12,-3,36,15
7,Q186,73,7,2,5,49,15
8,Q242,73,6,4,2,25,38
9,ALL_TARGETS,650,41,35,6,352,222


## Block 24. Automatic decision

Strict criteria for proceeding to the official test:

- Fast9 Macro-F1 is at least 0.02 higher than legacy;
- accuracy and alignment are not below legacy;
- coverage is 100%;
- at least 80% of the full alternative pipeline's Macro-F1 gain is retained;
- the F1 delta is positive for Q17, Q186, and Q242;
- paired net wins against legacy are positive.

In [24]:
critical_targets = [
    "Q17",
    "Q186",
    "Q242",
]

critical_target_rows = (
    score_comparison
    .set_index("question_id")
    .loc[critical_targets]
)

f1_retention = gain_retention(
    float(
        overall["legacy_macro_f1"]
    ),
    float(
        overall[
            "full_alternative_macro_f1"
        ]
    ),
    float(
        overall["fast9_macro_f1"]
    ),
)

all_wins = (
    win_loss_vs_legacy
    .query(
        "question_id == 'ALL_TARGETS'"
    )
    .iloc[0]
)

decision_criteria = {
    "macro_f1_delta_vs_legacy_at_least_0_02": (
        float(
            overall[
                "fast9_minus_legacy_macro_f1"
            ]
        )
        >= 0.02
    ),
    "accuracy_not_below_legacy": (
        float(
            overall[
                "fast9_minus_legacy_accuracy"
            ]
        )
        >= 0.0
    ),
    "alignment_not_below_legacy": (
        float(
            overall[
                "fast9_minus_legacy_alignment_proxy"
            ]
        )
        >= 0.0
    ),
    "coverage_is_100_percent": (
        float(
            overall["fast9_coverage"]
        )
        >= 1.0
    ),
    "retains_at_least_80_percent_of_full_f1_gain": (
        not pd.isna(f1_retention)
        and f1_retention >= 0.80
    ),
    "critical_targets_all_improve_over_legacy": (
        critical_target_rows[
            "fast9_minus_legacy_macro_f1"
        ].gt(0).all()
    ),
    "paired_net_wins_vs_legacy_positive": (
        int(all_wins["net_wins"])
        > 0
    ),
}

FAST9_DECISION = (
    "PROCEED TO OFFICIAL TEST"
    if all(decision_criteria.values())
    else "DO NOT RUN OFFICIAL TEST YET"
)

display(pd.DataFrame([
    {
        "criterion": name,
        "passed": bool(passed),
    }
    for name, passed
    in decision_criteria.items()
]))

print(
    "F1 gain retention:",
    (
        f"{f1_retention:.1%}"
        if not pd.isna(f1_retention)
        else "not defined"
    ),
)
print(FAST9_DECISION)


,criterion,passed
0,macro_f1_delta_vs_legacy_at_least_0_02,True
1,accuracy_not_below_legacy,True
2,alignment_not_below_legacy,True
3,coverage_is_100_percent,True
4,retains_at_least_80_percent_of_full_f1_gain,True
5,critical_targets_all_improve_over_legacy,True
6,paired_net_wins_vs_legacy_positive,True


F1 gain retention: 106.6%
PROCEED TO OFFICIAL TEST


## Block 25. Save validation results


In [25]:
timestamp = datetime.now().strftime(
    "%Y%m%d_%H%M%S"
)
VALIDATION_SAVE_DIR = Path(
    f"/content/fast9_strict_features_validation_{timestamp}"
)
VALIDATION_SAVE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

tables_to_save = {
    "reference_sample": reference_sample,
    "reference_legacy": reference_legacy,
    "reference_full_alternative": reference_full,
    "fast9_validation_result": (
        fast9_validation_result
    ),
    "score_comparison": score_comparison,
    "retention_summary": retention_summary,
    "bootstrap_vs_legacy": (
        bootstrap_vs_legacy
    ),
    "win_loss_vs_legacy": (
        win_loss_vs_legacy
    ),
    "bootstrap_vs_full": (
        bootstrap_vs_full
    ),
    "win_loss_vs_full": (
        win_loss_vs_full
    ),
    "feature_importance": (
        fast9_validation_pipeline
        .feature_importance_table
    ),
    "strict_feature_audit": strict_validation_feature_audit,
}

for name, frame in tables_to_save.items():
    frame.to_csv(
        VALIDATION_SAVE_DIR / f"{name}.csv",
        index=False,
    )

with (
    VALIDATION_SAVE_DIR
    / "config.json"
).open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        asdict(validation_cfg),
        file,
        ensure_ascii=False,
        indent=2,
        default=str,
    )

with (
    VALIDATION_SAVE_DIR
    / "decision.json"
).open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        {
            "decision": FAST9_DECISION,
            "criteria": {
                key: bool(value)
                for key, value
                in decision_criteria.items()
            },
            "f1_gain_retention": (
                None
                if pd.isna(f1_retention)
                else float(f1_retention)
            ),
        },
        file,
        ensure_ascii=False,
        indent=2,
    )

for source_path in Path(
    "/content"
).glob("*strict_features*.jsonl"):
    destination = (
        VALIDATION_SAVE_DIR
        / source_path.name
    )
    if (
        source_path.is_file()
        and source_path.resolve()
        != destination.resolve()
    ):
        shutil.copy2(
            source_path,
            destination,
        )

validation_archive = shutil.make_archive(
    str(VALIDATION_SAVE_DIR),
    "zip",
    VALIDATION_SAVE_DIR,
)

print(
    "Saved:",
    validation_archive,
)


Saved: /content/fast9_strict_features_validation_20260714_233149.zip


## Block 26. Download the validation archive

In [26]:
from google.colab import files

files.download(
    validation_archive
)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Part III. Official test

Run this section only after `PROCEED TO OFFICIAL TEST`.

For 1,050 respondents, there will be 9,450 primary target-specific calls. With `max_workers=48`, expected API wall time is usually around 40–100 minutes, although actual speed depends on Nebius rate limits and latency.

## Block 27. Fit Fast9 on the full training set

In [27]:
official_cfg = copy.deepcopy(
    CFG
)
official_cfg.cache_path = (
    "llm_cache_fast9_strict_features_official.jsonl"
)

official_llm = CachedNebiusClient(
    official_cfg,
    api_key=nebius_key,
    client=raw_nebius_client,
)

fast9_official_pipeline = (
    FastNineCallLLMPipeline(
        official_cfg,
        official_llm,
    )
    .fit(
        train,
        mode="mixed",
    )
)

print(
    "Full-train pipeline is ready."
)


strict_official_feature_audit = (
    audit_strict_feature_compliance(
        pipeline=fast9_official_pipeline,
        sample_row=test.iloc[0],
        test_df=test,
    )
)

display(
    strict_official_feature_audit
)


Loaded 0 cached LLM responses


Encoding feature columns:   0%|          | 0/278 [00:00<?, ?it/s]

Target feature selection:   0%|          | 0/9 [00:00<?, ?it/s]

Pipeline fit completed in 1.24 minutes.
Full-train pipeline is ready.
PASSED: model inputs are restricted to official feature variables. Country is used only for split construction and diagnostics.


,component,number_used,outside_official_feature_pool,absent_from_test,target_leakage,forbidden_identifiers,passed
0,hybrid_selection,86,[],[],[],[],True
1,catboost,104,[],[],[],[],True
2,retrieval,86,[],[],[],[],True


## Блок 28. Небольшой official preview

Эти 18 calls попадут в cache и повторно не оплачиваются при полном запуске.

In [28]:
official_preview_predictions, official_preview_raw = (
    run_fast9_predictions(
        pipeline=fast9_official_pipeline,
        input_df=test.head(2),
        run_id="official_strict_features_preview_2",
        include_truth=False,
    )
)

display(
    official_preview_raw[
        [
            "respondent_id",
            "question_id",
            "prediction",
            "llm_prediction",
            "catboost_prediction",
            "status",
            "repair_used",
            "margin",
        ]
    ]
)


New LLM jobs: 18; already checkpointed: 0


official_strict_features_preview_2:   0%|          | 0/18 [00:00<?, ?it/s]

Completed 18 predictions in 0.08 minutes.
Repair share: 0.00%
Fallback share: 0.00%


,respondent_id,question_id,prediction,llm_prediction,catboost_prediction,status,repair_used,margin
0,R20070701,Q17,Not so important,Not so important,Not so important,success_first_pass,False,0.437597
1,R20070477,Q17,Not so important,Not so important,Not so important,success_first_pass,False,0.371010
2,R20070477,Q209,Might do,Might do,Have done,success_first_pass,False,0.435292
3,R20070701,Q209,Have done,Have done,Have done,success_first_pass,False,0.857406
4,R20070701,Q148,Not at all,Not at all,Not at all,success_first_pass,False,0.927264
5,R20070477,Q201,Weekly,Weekly,Monthly,success_first_pass,False,0.040174
6,R20070701,Q73,A great deal,A great deal,A great deal,success_first_pass,False,0.396665
7,R20070477,Q227,Not often,Not often,Not often,success_first_pass,False,0.020536
8,R20070701,Q227,Not at all often,Not at all often,Not at all often,success_first_pass,False,0.952575
9,R20070701,Q201,Monthly,Monthly,Weekly,success_first_pass,False,0.012030


## Block 29. Full official inference


In [29]:
official_started = time.perf_counter()

official_predictions, official_raw = (
    run_fast9_predictions(
        pipeline=fast9_official_pipeline,
        input_df=test,
        run_id="official_strict_features_full",
        include_truth=False,
    )
)

official_minutes = (
    time.perf_counter()
    - official_started
) / 60

print(
    "Predictions:",
    official_predictions.shape,
)
print(
    "Coverage:",
    len(official_predictions)
    / (len(test) * len(target_ids)),
)
print(
    "Repair share:",
    official_raw[
        "repair_used"
    ].mean(),
)
print(
    "Fallback share:",
    official_raw[
        "status"
    ].str.contains(
        "fallback"
    ).mean(),
)
print(
    f"Official wall time: "
    f"{official_minutes:.2f} minutes."
)

display(
    official_predictions.head(20)
)


New LLM jobs: 9,450; already checkpointed: 0


official_strict_features_full:   0%|          | 0/9450 [00:00<?, ?it/s]

Completed 9,450 predictions in 18.19 minutes.
Repair share: 0.00%
Fallback share: 0.00%
Predictions: (9450, 3)
Coverage: 1.0
Repair share: 0.0
Fallback share: 0.0
Official wall time: 18.19 minutes.


,respondent_id,question_id,prediction
0,R20070701,Q73,A great deal
1,R20070477,Q148,A great deal
2,R20070477,Q33,Disagree strongly
3,R20070701,Q227,Not at all often
4,R20070477,Q73,Quite a lot
5,R20070477,Q242,Somewhat unimportant
6,R20070477,Q201,Weekly
7,R20070701,Q201,Monthly
8,R20070701,Q186,Always justifiable
9,R20070477,Q209,Might do


## Block 30. Create the submission ZIP

In [30]:
def write_fast9_submission(
    predictions: pd.DataFrame,
    pipeline: FastNineCallLLMPipeline,
    run_id: str = "group6_fast9_llm_primary_strict_features",
) -> Path:
    submission_dir = Path(
        f"submission_{run_id}"
    )
    shutil.rmtree(
        submission_dir,
        ignore_errors=True,
    )
    (
        submission_dir
        / "method"
    ).mkdir(
        parents=True,
        exist_ok=True,
    )

    predictions.to_csv(
        submission_dir
        / "predictions.csv",
        index=False,
    )

    disclosure_rows = []

    for target_id in target_ids:
        used_features = set(
            pipeline.hybrid_selected[
                target_id
            ]
        )
        used_features.update(
            pipeline.expert.features.get(
                target_id,
                [],
            )
        )

        for variable in sorted(
            used_features
        ):
            if variable in candidate_features:
                disclosure_rows.append({
                    "question_id": target_id,
                    "feature_variable_code": variable,
                })

    disclosure_df = pd.DataFrame(
        disclosure_rows
    ).drop_duplicates()

    disclosed_features = set(
        disclosure_df[
            "feature_variable_code"
        ].astype(str)
    )

    assert disclosed_features <= OFFICIAL_FEATURE_POOL
    assert "country" not in disclosed_features
    assert "respondent_id" not in disclosed_features
    assert not (
        disclosed_features & target_id_set
    )

    disclosure_df.to_csv(
        submission_dir
        / "features.csv",
        index=False,
    )

    example_row = test.iloc[0]

    with (
        submission_dir
        / "method"
        / "prompts.jsonl"
    ).open(
        "w",
        encoding="utf-8",
    ) as file:
        for target_id in target_ids:
            record = {
                "question_id": target_id,
                "model": (
                    pipeline.config.model
                ),
                "prompt_features": (
                    pipeline.hybrid_selected[
                        target_id
                    ]
                ),
                "prompt": (
                    pipeline.preview_prompt(
                        example_row,
                        target_id,
                    )
                ),
            }
            file.write(
                json.dumps(
                    record,
                    ensure_ascii=False,
                )
                + "\n"
            )

    method_text = f"""
Model: {pipeline.config.model}

The method is an LLM-primary target-specific survey-response pipeline.
It makes one LLM call for each of the nine target questions for every
respondent. Feature selection is fit-only and leakage-safe.

All predictive survey inputs are restricted to variables listed in the
official features dataset. The separate country and respondent_id columns
are not used for feature selection, CatBoost prediction, retrieval similarity
or LLM prompting. Country is used only to construct validation splits and
country-clustered evaluation diagnostics; respondent_id is used only as a
technical key for caching and checkpoints.

For each target, normalized mutual information and mRMR first identify
relevant non-target survey answers. CatBoost is trained on an expanded
feature pool and its feature importance is combined conservatively with
the NMI/mRMR ranking. The first eight mRMR-selected features are protected,
up to four nonlinear CatBoost-important features are added, and a compact
target-specific evidence set is formed.

Each LLM prompt contains up to {pipeline.config.prompt_features} direct
respondent answers, {pipeline.config.n_retrieval_examples} compact labelled
training analogues, a target-specific guide, exact official labels and a
fit-only CatBoost probability prior. Q186 and Q242 are modeled directly in
their official coarsened label spaces.

The LLM returns a probability distribution in JSON. The final distribution
uses 88% LLM probabilities and 12% CatBoost probabilities. There is no
latent-profile call and no ordinary reasoning second pass. A short repair
call is used only when the structured output is invalid. CatBoost is used
as a final fallback after technical failure, ensuring complete coverage.

API calls are parallelized with {pipeline.config.max_workers} workers and
all responses are cached and checkpointed.
""".strip() + "\n"

    (
        submission_dir
        / "method"
        / "method.md"
    ).write_text(
        method_text,
        encoding="utf-8",
    )

    archive_path = shutil.make_archive(
        submission_dir.name,
        "zip",
        root_dir=".",
        base_dir=submission_dir.name,
    )

    return Path(
        archive_path
    )


submission_zip = write_fast9_submission(
    official_predictions,
    fast9_official_pipeline,
)

print(
    "Created:",
    submission_zip,
)


Created: /content/submission_group6_fast9_llm_primary_strict_features.zip


In [31]:
from google.colab import files

files.download(
    str(submission_zip)
)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>